# WBSNet Research Notebook

This notebook is a **paper-oriented, self-contained Kaggle notebook** for the WBSNet project.

It does not import from the local `wbsnet/` package. The notebook implements:
- the processed-dataset loader for `WBSNet_Dataset`
- the full WBSNet architecture
- ablations `A1` to `A7`
- training, validation, evaluation, checkpointing, and qualitative exports
- result aggregation, significance testing, complexity estimation, lambda sweeps, and paper-ready tables

The notebook is designed for two workflows:
1. **Single-run workflow** for debugging one model on one dataset.
2. **Full paper workflow** for running the full ablation suite across seeds and datasets, then generating tables and figures.


In [ ]:
# Lightweight dependencies used by the research analysis cells.
!pip install -q pywavelets scipy


## Notebook Roadmap

Recommended execution order:

1. Review the config cell and dataset auto-detection.
2. Run the data-pipeline and model-definition cells.
3. Run the **single-experiment** cell first for a sanity check.
4. When that works, enable the **paper-suite** cell to launch the multi-seed ablations.
5. Run the analysis cells to build the paper tables, significance reports, qualitative panels, and complexity summaries.

Notes:
- Long-running cells are guarded by boolean flags so the notebook does not accidentally start a full paper sweep.
- The notebook can merge your locally produced results with paper-registered external baselines from the manuscript.


In [ ]:
from __future__ import annotations

import copy
import json
import math
import os
import random
import re
import time
import warnings
from collections import defaultdict
from dataclasses import dataclass, field
from functools import lru_cache
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from PIL import Image, ImageDraw, ImageEnhance
from scipy import stats
from torch.cuda.amp import GradScaler as CudaGradScaler
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

warnings.filterwarnings("once")


def create_grad_scaler(device: torch.device, enabled: bool) -> Any:
    enabled = bool(enabled and device.type == "cuda")
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        return torch.amp.GradScaler(device.type, enabled=enabled)
    return CudaGradScaler(enabled=enabled)


def seed_everything(seed: int, deterministic: bool = False) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def save_json(path: str | Path, payload: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def load_json(path: str | Path) -> Any:
    return json.loads(Path(path).read_text(encoding="utf-8"))


def natural_sort_key(value: str) -> list[object]:
    return [int(token) if token.isdigit() else token.lower() for token in re.split(r"(\d+)", value)]


def autodetect_processed_root(explicit_root: str | None = None) -> Path:
    if explicit_root:
        root = Path(explicit_root).expanduser()
        if not root.exists():
            raise FileNotFoundError(f"Configured processed_root does not exist: {root}")
        return root

    candidates: list[Path] = []
    for base in [Path("/kaggle/input"), Path.cwd()]:
        if not base.exists():
            continue

        direct = base / "WBSNet_Dataset"
        if (direct / "dataset_info.json").exists():
            candidates.append(direct)

        for child in sorted(base.iterdir()):
            if not child.is_dir():
                continue
            if child.name == "WBSNet_Dataset" and (child / "dataset_info.json").exists():
                candidates.append(child)
            nested = child / "WBSNet_Dataset"
            if (nested / "dataset_info.json").exists():
                candidates.append(nested)

    if not candidates:
        raise FileNotFoundError(
            "Could not auto-detect WBSNet_Dataset. Set RESEARCH_CONFIG['processed_root'] manually."
        )

    return Path(sorted({str(path.resolve()) for path in candidates})[0])


VARIANT_SPECS = {
    "A1": {
        "display_name": "Identity Skip U-Net",
        "paper_name": "U-Net (ResNet-34)",
        "overrides": {
            "use_wavelet": False,
            "use_lfsa": False,
            "use_hfba": False,
            "boundary_supervision": False,
        },
    },
    "A2": {
        "display_name": "WBSNet Full",
        "paper_name": "WBSNet (A2)",
        "overrides": {
            "use_wavelet": True,
            "use_lfsa": True,
            "use_hfba": True,
            "boundary_supervision": True,
            "wavelet_type": "haar",
        },
    },
    "A3": {
        "display_name": "LFSA Only",
        "paper_name": "A3 LFSA Only",
        "overrides": {
            "use_wavelet": True,
            "use_lfsa": True,
            "use_hfba": False,
            "boundary_supervision": False,
            "wavelet_type": "haar",
        },
    },
    "A4": {
        "display_name": "HFBA Only",
        "paper_name": "A4 HFBA Only",
        "overrides": {
            "use_wavelet": True,
            "use_lfsa": False,
            "use_hfba": True,
            "boundary_supervision": True,
            "wavelet_type": "haar",
        },
    },
    "A5": {
        "display_name": "No Boundary Supervision",
        "paper_name": "A5 No Boundary Supervision",
        "overrides": {
            "use_wavelet": True,
            "use_lfsa": True,
            "use_hfba": True,
            "boundary_supervision": False,
            "wavelet_type": "haar",
        },
    },
    "A6": {
        "display_name": "No Wavelet Attention",
        "paper_name": "A6 No Wavelet",
        "overrides": {
            "use_wavelet": False,
            "use_lfsa": True,
            "use_hfba": True,
            "boundary_supervision": True,
        },
    },
    "A7": {
        "display_name": "db2 Wavelet",
        "paper_name": "A7 db2 Wavelet",
        "overrides": {
            "use_wavelet": True,
            "use_lfsa": True,
            "use_hfba": True,
            "boundary_supervision": True,
            "wavelet_type": "db2",
        },
    },
}

DATASET_SPECS = {
    "kvasir": {
        "display_name": "Kvasir-SEG",
        "epochs": 200,
        "paper_eval_split": "test",
        "fallback_eval_split": "val",
        "cross_dataset_targets": [{"dataset": "cvc_colondb", "split": "test"}],
    },
    "cvc_clinicdb": {
        "display_name": "CVC-ClinicDB",
        "epochs": 200,
        "paper_eval_split": "test",
        "fallback_eval_split": "val",
        "cross_dataset_targets": [],
    },
    "cvc_colondb": {
        "display_name": "CVC-ColonDB",
        "epochs": 1,
        "paper_eval_split": "test",
        "fallback_eval_split": "test",
        "cross_dataset_targets": [],
    },
    "isic2018": {
        "display_name": "ISIC 2018",
        "epochs": 100,
        "paper_eval_split": "test",
        "fallback_eval_split": "val",
        "cross_dataset_targets": [],
    },
}

PAPER_REPORTED_MAIN = {
    "kvasir": {
        "U-Net (ResNet-34)": {"dice_mean": 0.881, "dice_std": 0.004, "iou_mean": 0.816, "iou_std": 0.005, "hd95_mean": 14.7, "hd95_std": 0.6},
        "U-Net++": {"dice_mean": 0.889, "dice_std": 0.004, "iou_mean": 0.824, "iou_std": 0.005, "hd95_mean": 13.4, "hd95_std": 0.5},
        "Attention U-Net": {"dice_mean": 0.893, "dice_std": 0.005, "iou_mean": 0.828, "iou_std": 0.006, "hd95_mean": 13.0, "hd95_std": 0.5},
        "U-Net v2": {"dice_mean": 0.905, "dice_std": 0.003, "iou_mean": 0.842, "iou_std": 0.004, "hd95_mean": 11.9, "hd95_std": 0.4},
        "TransUNet": {"dice_mean": 0.908, "dice_std": 0.004, "iou_mean": 0.845, "iou_std": 0.005, "hd95_mean": 11.7, "hd95_std": 0.4},
        "Swin-UNet": {"dice_mean": 0.906, "dice_std": 0.004, "iou_mean": 0.843, "iou_std": 0.005, "hd95_mean": 11.9, "hd95_std": 0.4},
        "PraNet": {"dice_mean": 0.898, "dice_std": 0.004, "iou_mean": 0.840, "iou_std": 0.005, "hd95_mean": 12.4, "hd95_std": 0.5},
        "MEGANet": {"dice_mean": 0.912, "dice_std": 0.003, "iou_mean": 0.851, "iou_std": 0.004, "hd95_mean": 10.9, "hd95_std": 0.4},
        "BMANet": {"dice_mean": 0.914, "dice_std": 0.003, "iou_mean": 0.852, "iou_std": 0.004, "hd95_mean": 10.7, "hd95_std": 0.4},
        "VM-UNet": {"dice_mean": 0.906, "dice_std": 0.003, "iou_mean": 0.843, "iou_std": 0.004, "hd95_mean": 11.8, "hd95_std": 0.4},
        "U-Mamba": {"dice_mean": 0.911, "dice_std": 0.004, "iou_mean": 0.849, "iou_std": 0.005, "hd95_mean": 11.1, "hd95_std": 0.4},
        "WBSNet (A2)": {"dice_mean": 0.923, "dice_std": 0.003, "iou_mean": 0.865, "iou_std": 0.004, "hd95_mean": 9.6, "hd95_std": 0.3},
    },
    "cvc_clinicdb": {
        "U-Net (ResNet-34)": {"dice_mean": 0.901, "dice_std": 0.005, "iou_mean": 0.841, "iou_std": 0.006, "hd95_mean": 11.9, "hd95_std": 0.5},
        "U-Net++": {"dice_mean": 0.908, "dice_std": 0.004, "iou_mean": 0.849, "iou_std": 0.005, "hd95_mean": 11.2, "hd95_std": 0.4},
        "Attention U-Net": {"dice_mean": 0.912, "dice_std": 0.004, "iou_mean": 0.854, "iou_std": 0.005, "hd95_mean": 10.6, "hd95_std": 0.4},
        "U-Net v2": {"dice_mean": 0.919, "dice_std": 0.003, "iou_mean": 0.864, "iou_std": 0.004, "hd95_mean": 10.0, "hd95_std": 0.3},
        "TransUNet": {"dice_mean": 0.921, "dice_std": 0.004, "iou_mean": 0.866, "iou_std": 0.005, "hd95_mean": 9.8, "hd95_std": 0.4},
        "Swin-UNet": {"dice_mean": 0.917, "dice_std": 0.004, "iou_mean": 0.862, "iou_std": 0.005, "hd95_mean": 10.2, "hd95_std": 0.4},
        "PraNet": {"dice_mean": 0.899, "dice_std": 0.005, "iou_mean": 0.849, "iou_std": 0.006, "hd95_mean": 10.7, "hd95_std": 0.4},
        "MEGANet": {"dice_mean": 0.925, "dice_std": 0.003, "iou_mean": 0.870, "iou_std": 0.004, "hd95_mean": 9.5, "hd95_std": 0.3},
        "BMANet": {"dice_mean": 0.925, "dice_std": 0.003, "iou_mean": 0.871, "iou_std": 0.004, "hd95_mean": 9.4, "hd95_std": 0.3},
        "VM-UNet": {"dice_mean": 0.918, "dice_std": 0.003, "iou_mean": 0.864, "iou_std": 0.004, "hd95_mean": 10.1, "hd95_std": 0.3},
        "U-Mamba": {"dice_mean": 0.922, "dice_std": 0.003, "iou_mean": 0.868, "iou_std": 0.004, "hd95_mean": 9.7, "hd95_std": 0.3},
        "WBSNet (A2)": {"dice_mean": 0.934, "dice_std": 0.003, "iou_mean": 0.882, "iou_std": 0.004, "hd95_mean": 8.4, "hd95_std": 0.3},
    },
    "isic2018": {
        "U-Net (ResNet-34)": {"dice_mean": 0.882, "dice_std": 0.004, "iou_mean": 0.813, "iou_std": 0.005, "hd95_mean": 17.8, "hd95_std": 0.7},
        "U-Net++": {"dice_mean": 0.886, "dice_std": 0.004, "iou_mean": 0.817, "iou_std": 0.005, "hd95_mean": 16.7, "hd95_std": 0.6},
        "Attention U-Net": {"dice_mean": 0.890, "dice_std": 0.004, "iou_mean": 0.822, "iou_std": 0.005, "hd95_mean": 16.1, "hd95_std": 0.6},
        "U-Net v2": {"dice_mean": 0.897, "dice_std": 0.003, "iou_mean": 0.830, "iou_std": 0.004, "hd95_mean": 15.2, "hd95_std": 0.5},
        "TransUNet": {"dice_mean": 0.898, "dice_std": 0.003, "iou_mean": 0.831, "iou_std": 0.004, "hd95_mean": 15.0, "hd95_std": 0.5},
        "Swin-UNet": {"dice_mean": 0.894, "dice_std": 0.004, "iou_mean": 0.827, "iou_std": 0.005, "hd95_mean": 15.4, "hd95_std": 0.5},
        "MEGANet": {"dice_mean": 0.902, "dice_std": 0.003, "iou_mean": 0.838, "iou_std": 0.004, "hd95_mean": 14.3, "hd95_std": 0.4},
        "WA-NET": {"dice_mean": 0.923, "dice_std": 0.003, "iou_mean": 0.862, "iou_std": 0.004, "hd95_mean": 13.1, "hd95_std": 0.4},
        "VM-UNet": {"dice_mean": 0.900, "dice_std": 0.003, "iou_mean": 0.834, "iou_std": 0.004, "hd95_mean": 14.8, "hd95_std": 0.5},
        "U-Mamba": {"dice_mean": 0.901, "dice_std": 0.003, "iou_mean": 0.836, "iou_std": 0.004, "hd95_mean": 14.5, "hd95_std": 0.4},
        "WBSNet (A2)": {"dice_mean": 0.927, "dice_std": 0.003, "iou_mean": 0.866, "iou_std": 0.004, "hd95_mean": 12.7, "hd95_std": 0.4},
    },
}

PAPER_REPORTED_CROSS_DATASET = {
    "kvasir_to_cvc_colondb": {
        "U-Net (ResNet-34)": {"dice_mean": 0.712, "dice_std": 0.008, "iou_mean": 0.611, "iou_std": 0.010, "hd95_mean": 31.4, "hd95_std": 1.1},
        "U-Net v2": {"dice_mean": 0.748, "dice_std": 0.007, "iou_mean": 0.648, "iou_std": 0.009, "hd95_mean": 27.2, "hd95_std": 1.0},
        "MEGANet": {"dice_mean": 0.766, "dice_std": 0.007, "iou_mean": 0.668, "iou_std": 0.008, "hd95_mean": 25.1, "hd95_std": 0.9},
        "U-Mamba": {"dice_mean": 0.771, "dice_std": 0.006, "iou_mean": 0.675, "iou_std": 0.008, "hd95_mean": 24.6, "hd95_std": 0.9},
        "WBSNet (A2)": {"dice_mean": 0.789, "dice_std": 0.006, "iou_mean": 0.697, "iou_std": 0.007, "hd95_mean": 22.4, "hd95_std": 0.8},
    }
}

PAPER_REPORTED_COMPLEXITY = {
    "U-Net (ResNet-34)": {"params_m": 24.4, "gflops": 57.1, "dice": 0.881, "hd95": 14.7},
    "Attention U-Net": {"params_m": 25.6, "gflops": 59.2, "dice": 0.893, "hd95": 13.0},
    "U-Net v2": {"params_m": 27.1, "gflops": 62.8, "dice": 0.905, "hd95": 11.9},
    "TransUNet": {"params_m": 93.2, "gflops": 132.0, "dice": 0.908, "hd95": 11.7},
    "Swin-UNet": {"params_m": 41.4, "gflops": 82.3, "dice": 0.906, "hd95": 11.9},
    "MEGANet": {"params_m": 28.3, "gflops": 64.1, "dice": 0.912, "hd95": 10.9},
    "U-Mamba": {"params_m": 41.0, "gflops": 76.3, "dice": 0.911, "hd95": 11.1},
    "WBSNet (A2)": {"params_m": 25.5, "gflops": 63.2, "dice": 0.923, "hd95": 9.6},
}

default_output_root = "/kaggle/working/wbsnet_paper_runs" if Path("/kaggle/working").exists() else "./wbsnet_paper_runs"
default_num_workers = 0 if Path("/kaggle").exists() else 2
default_pin_memory = False if Path("/kaggle").exists() else True

RESEARCH_CONFIG = {
    "processed_root": None,
    "output_root": default_output_root,
    "image_size": [352, 352],
    "batch_size": 4,
    "num_workers": default_num_workers,
    "pin_memory": default_pin_memory,
    "normalize_mean": [0.485, 0.456, 0.406],
    "normalize_std": [0.229, 0.224, 0.225],
    "augment": {
        "random_resized_crop": True,
        "random_resized_crop_scale": [0.8, 1.0],
        "random_resized_crop_ratio": [0.9, 1.1],
        "horizontal_flip": True,
        "vertical_flip": True,
        "rotate90": True,
        "color_jitter": True,
    },
    "decoder_channels": [256, 128, 64, 32],
    "reduction_ratio": 16,
    "encoder_pretrained": True,
    "allow_pretrained_fallback": True,
    "amp": True,
    "compile_model": False,
    "encoder_lr": 1e-4,
    "decoder_lr": 1e-3,
    "weight_decay": 1e-4,
    "grad_accum_steps": 1,
    "clip_grad_norm": 1.0,
    "threshold": 0.5,
    "compute_hd95": True,
    "max_visualizations": 12,
    "contact_sheet_columns": 2,
    "deterministic": False,
    "resume_training": True,
    "resume_evaluation": True,
    "tolerate_export_failures": True,
    "single_run_dataset": "kvasir",
    "single_run_variant": "A2",
    "single_run_seed": 3407,
    "single_run_epochs_override": None,
    "paper_suite_datasets": ["kvasir", "cvc_clinicdb", "isic2018"],
    "paper_suite_variants": ["A1", "A2", "A3", "A4", "A5", "A6", "A7"],
    "paper_suite_seeds": [3407, 3408, 3409],
    "lambda_sweep_values": [0.1, 0.3, 0.5, 1.0],
    "size_breakdown_reference_variants": ["A1", "A2"],
}

PROCESSED_ROOT = autodetect_processed_root(RESEARCH_CONFIG["processed_root"]).resolve()
OUTPUT_ROOT = Path(RESEARCH_CONFIG["output_root"]).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATASET_INFO_PATH = PROCESSED_ROOT / "dataset_info.json"
DATASET_INFO = load_json(DATASET_INFO_PATH) if DATASET_INFO_PATH.exists() else {}
AVAILABLE_DATASETS = sorted(path.name for path in PROCESSED_ROOT.iterdir() if path.is_dir())

print(f"Processed dataset root: {PROCESSED_ROOT}")
print(f"Available datasets: {AVAILABLE_DATASETS}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Device: {DEVICE}")
if DATASET_INFO:
    display(pd.DataFrame(DATASET_INFO.get("datasets", {})).T)


## Data Pipeline

The next cell defines the processed dataset loader used by all experiments.

The loader:
- reads `images`, `masks`, and `boundaries` from the processed Kaggle dataset
- applies the same geometric transform to all three tensors
- keeps masks and boundaries binary
- returns everything needed for training, evaluation, qualitative export, and per-sample analysis


In [ ]:
RESAMPLING = Image.Resampling


def list_file_map(directory: str | Path, suffixes: tuple[str, ...] = (".png", ".jpg", ".jpeg")) -> dict[str, Path]:
    directory = Path(directory)
    files = [path for path in directory.iterdir() if path.is_file() and path.suffix.lower() in suffixes]
    return {path.stem: path for path in sorted(files, key=lambda item: natural_sort_key(item.name))}


def discover_processed_samples(processed_root: str | Path, dataset_name: str, split: str) -> list[dict[str, Any]]:
    split_root = Path(processed_root) / dataset_name / split
    image_dir = split_root / "images"
    mask_dir = split_root / "masks"
    boundary_dir = split_root / "boundaries"

    if not image_dir.exists() or not mask_dir.exists() or not boundary_dir.exists():
        raise FileNotFoundError(f"Missing processed split directories for {dataset_name}/{split}: {split_root}")

    image_map = list_file_map(image_dir)
    mask_map = list_file_map(mask_dir)
    boundary_map = list_file_map(boundary_dir)
    shared_keys = sorted(set(image_map) & set(mask_map) & set(boundary_map), key=natural_sort_key)

    return [
        {
            "sample_id": sample_id,
            "image_path": image_map[sample_id],
            "mask_path": mask_map[sample_id],
            "boundary_path": boundary_map[sample_id],
        }
        for sample_id in shared_keys
    ]


@dataclass
class SegmentationTransform:
    image_size: tuple[int, int]
    train: bool
    mean: tuple[float, float, float]
    std: tuple[float, float, float]
    augment: dict[str, Any]

    def __call__(self, image: Image.Image, mask: Image.Image, boundary: Image.Image) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        image = image.convert("RGB")
        mask = mask.convert("L")
        boundary = boundary.convert("L")

        if self.train:
            image, mask, boundary = self._augment(image, mask, boundary)

        target_size = (self.image_size[1], self.image_size[0])
        image = image.resize(target_size, resample=RESAMPLING.BILINEAR)
        mask = mask.resize(target_size, resample=RESAMPLING.NEAREST)
        boundary = boundary.resize(target_size, resample=RESAMPLING.NEAREST)

        image_np = np.asarray(image, dtype=np.float32) / 255.0
        image_np = (image_np - np.array(self.mean, dtype=np.float32)) / np.array(self.std, dtype=np.float32)
        mask_np = (np.asarray(mask, dtype=np.float32) > 127).astype(np.float32)
        boundary_np = (np.asarray(boundary, dtype=np.float32) > 127).astype(np.float32)

        image_tensor = torch.from_numpy(image_np.transpose(2, 0, 1)).float()
        mask_tensor = torch.from_numpy(mask_np).unsqueeze(0).float()
        boundary_tensor = torch.from_numpy(boundary_np).unsqueeze(0).float()
        return image_tensor, mask_tensor, boundary_tensor

    def _augment(
        self,
        image: Image.Image,
        mask: Image.Image,
        boundary: Image.Image,
    ) -> tuple[Image.Image, Image.Image, Image.Image]:
        if self.augment.get("random_resized_crop", False):
            image, mask, boundary = self._random_resized_crop(image, mask, boundary)
        if self.augment.get("horizontal_flip", False) and random.random() < 0.5:
            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            mask = mask.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            boundary = boundary.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        if self.augment.get("vertical_flip", False) and random.random() < 0.5:
            image = image.transpose(Image.Transpose.FLIP_TOP_BOTTOM)
            mask = mask.transpose(Image.Transpose.FLIP_TOP_BOTTOM)
            boundary = boundary.transpose(Image.Transpose.FLIP_TOP_BOTTOM)
        if self.augment.get("rotate90", False) and random.random() < 0.5:
            rotation = random.choice(
                [
                    Image.Transpose.ROTATE_90,
                    Image.Transpose.ROTATE_180,
                    Image.Transpose.ROTATE_270,
                ]
            )
            image = image.transpose(rotation)
            mask = mask.transpose(rotation)
            boundary = boundary.transpose(rotation)
        if self.augment.get("color_jitter", False):
            image = self._color_jitter(image)
        return image, mask, boundary

    @staticmethod
    def _color_jitter(image: Image.Image) -> Image.Image:
        for enhancer_cls in (ImageEnhance.Brightness, ImageEnhance.Contrast, ImageEnhance.Color):
            image = enhancer_cls(image).enhance(random.uniform(0.9, 1.1))
        return image

    def _random_resized_crop(
        self,
        image: Image.Image,
        mask: Image.Image,
        boundary: Image.Image,
    ) -> tuple[Image.Image, Image.Image, Image.Image]:
        width, height = image.size
        area = width * height
        scale = tuple(self.augment.get("random_resized_crop_scale", [0.8, 1.0]))
        ratio = tuple(self.augment.get("random_resized_crop_ratio", [0.9, 1.1]))
        log_ratio = (math.log(ratio[0]), math.log(ratio[1]))

        for _ in range(10):
            target_area = random.uniform(scale[0], scale[1]) * area
            aspect_ratio = math.exp(random.uniform(*log_ratio))
            crop_width = int(round(math.sqrt(target_area * aspect_ratio)))
            crop_height = int(round(math.sqrt(target_area / aspect_ratio)))
            if 0 < crop_width <= width and 0 < crop_height <= height:
                left = random.randint(0, width - crop_width)
                top = random.randint(0, height - crop_height)
                box = (left, top, left + crop_width, top + crop_height)
                return image.crop(box), mask.crop(box), boundary.crop(box)

        min_side = min(width, height)
        left = (width - min_side) // 2
        top = (height - min_side) // 2
        box = (left, top, left + min_side, top + min_side)
        return image.crop(box), mask.crop(box), boundary.crop(box)


class ProcessedWBSNetDataset(Dataset):
    def __init__(
        self,
        processed_root: str | Path,
        dataset_name: str,
        split: str,
        train: bool,
        config: dict[str, Any],
    ) -> None:
        self.processed_root = Path(processed_root)
        self.dataset_name = dataset_name
        self.split = split
        self.records = discover_processed_samples(self.processed_root, dataset_name, split)
        self.transform = SegmentationTransform(
            image_size=tuple(config["image_size"]),
            train=train,
            mean=tuple(config["normalize_mean"]),
            std=tuple(config["normalize_std"]),
            augment=config["augment"],
        )

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> dict[str, Any]:
        record = self.records[index]
        with Image.open(record["image_path"]) as image:
            with Image.open(record["mask_path"]) as mask:
                with Image.open(record["boundary_path"]) as boundary:
                    image_tensor, mask_tensor, boundary_tensor = self.transform(image, mask, boundary)
        return {
            "image": image_tensor,
            "mask": mask_tensor,
            "boundary": boundary_tensor,
            "sample_id": record["sample_id"],
            "image_path": str(record["image_path"]),
            "mask_path": str(record["mask_path"]),
            "boundary_path": str(record["boundary_path"]),
        }


def build_dataloader(dataset: Dataset, batch_size: int, shuffle: bool, config: dict[str, Any]) -> DataLoader:
    num_workers = int(config["num_workers"])
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=bool(config["pin_memory"] and torch.cuda.is_available()),
        persistent_workers=bool(num_workers > 0),
        drop_last=shuffle,
    )


def available_splits(dataset_name: str) -> list[str]:
    dataset_root = PROCESSED_ROOT / dataset_name
    return sorted(path.name for path in dataset_root.iterdir() if path.is_dir())


def resolve_eval_split(dataset_name: str) -> str:
    requested = DATASET_SPECS[dataset_name]["paper_eval_split"]
    splits = set(available_splits(dataset_name))
    if requested in splits:
        return requested
    fallback = DATASET_SPECS[dataset_name]["fallback_eval_split"]
    if fallback in splits:
        return fallback
    if "val" in splits:
        return "val"
    if "test" in splits:
        return "test"
    raise ValueError(f"No suitable evaluation split found for {dataset_name}. Available: {sorted(splits)}")


def denormalize_image_tensor(image: torch.Tensor, mean: list[float], std: list[float]) -> np.ndarray:
    image_np = image.detach().cpu().numpy().transpose(1, 2, 0)
    mean_np = np.array(mean, dtype=np.float32)
    std_np = np.array(std, dtype=np.float32)
    image_np = (image_np * std_np) + mean_np
    image_np = np.clip(image_np * 255.0, 0, 255).astype(np.uint8)
    return image_np


def tensor_to_mask_uint8(mask: torch.Tensor) -> np.ndarray:
    mask_np = mask.detach().cpu().numpy()
    if mask_np.ndim == 3:
        mask_np = mask_np[0]
    return (mask_np > 0.5).astype(np.uint8) * 255


def show_samples(dataset: Dataset, count: int = 3, title: str = "Preview") -> None:
    count = min(count, len(dataset))
    if count == 0:
        print(f"No samples available for {title}.")
        return
    fig, axes = plt.subplots(count, 3, figsize=(12, 4 * count))
    if count == 1:
        axes = np.expand_dims(axes, axis=0)
    for row in range(count):
        sample = dataset[row]
        image_np = denormalize_image_tensor(sample["image"], RESEARCH_CONFIG["normalize_mean"], RESEARCH_CONFIG["normalize_std"])
        mask_np = tensor_to_mask_uint8(sample["mask"])
        boundary_np = tensor_to_mask_uint8(sample["boundary"])
        axes[row, 0].imshow(image_np)
        axes[row, 0].set_title(f"{title} | {sample['sample_id']} | image")
        axes[row, 1].imshow(mask_np, cmap="gray", vmin=0, vmax=255)
        axes[row, 1].set_title("mask")
        axes[row, 2].imshow(boundary_np, cmap="gray", vmin=0, vmax=255)
        axes[row, 2].set_title("boundary")
        for axis in axes[row]:
            axis.axis("off")
    plt.tight_layout()
    plt.show()


## WBSNet Architecture and Variants

The next two cells define:
- the wavelet operators
- LFSA and HFBA
- the ResNet-34-style encoder
- the U-Net-style decoder
- the full WBSNet model
- variant overrides for `A1` to `A7`


In [ ]:
class LFSA(nn.Module):
    def __init__(self, channels: int, reduction_ratio: int = 16) -> None:
        super().__init__()
        reduced = max(channels // reduction_ratio, 4)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, reduced, kernel_size=1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(reduced, channels, kernel_size=1, bias=True),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.fc(self.pool(x))


class HFBA(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        merged_channels = channels * 3
        self.depthwise = nn.Sequential(
            nn.Conv2d(merged_channels, merged_channels, kernel_size=3, padding=1, groups=merged_channels, bias=False),
            nn.BatchNorm2d(merged_channels),
            nn.ReLU(inplace=True),
        )
        self.pointwise = nn.Sequential(
            nn.Conv2d(merged_channels, channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )
        self.boundary_head = nn.Conv2d(channels, 1, kernel_size=1)

    def forward(self, lh: torch.Tensor, hl: torch.Tensor, hh: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        merged = torch.cat([lh, hl, hh], dim=1)
        reduced = self.pointwise(self.depthwise(merged))
        boundary_logits = self.boundary_head(reduced)
        attended = reduced * torch.sigmoid(boundary_logits)
        return attended, boundary_logits


def _haar_filters() -> tuple[list[float], list[float], list[float], list[float]]:
    scale = 2 ** -0.5
    dec_lo = [scale, scale]
    dec_hi = [-scale, scale]
    rec_lo = [scale, scale]
    rec_hi = [scale, -scale]
    return dec_lo, dec_hi, rec_lo, rec_hi


@lru_cache(maxsize=8)
def _wavelet_filters(wavelet_type: str) -> tuple[list[float], list[float], list[float], list[float]]:
    wavelet_type = str(wavelet_type).lower()
    if wavelet_type == "haar":
        return _haar_filters()
    try:
        import pywt

        wavelet = pywt.Wavelet(wavelet_type)
        return list(wavelet.dec_lo), list(wavelet.dec_hi), list(wavelet.rec_lo), list(wavelet.rec_hi)
    except Exception as exc:
        raise NotImplementedError(f"Wavelet '{wavelet_type}' requires pywavelets or a manual implementation.") from exc


def _stack_filters(lo: list[float], hi: list[float], device: torch.device, dtype: torch.dtype) -> torch.Tensor:
    lo_tensor = torch.tensor(lo, device=device, dtype=dtype)
    hi_tensor = torch.tensor(hi, device=device, dtype=dtype)
    ll = torch.outer(lo_tensor, lo_tensor)
    lh = torch.outer(lo_tensor, hi_tensor)
    hl = torch.outer(hi_tensor, lo_tensor)
    hh = torch.outer(hi_tensor, hi_tensor)
    return torch.stack([ll, lh, hl, hh], dim=0).unsqueeze(1)


class WaveletTransform2d(nn.Module):
    def __init__(self, wavelet_type: str = "haar") -> None:
        super().__init__()
        self.wavelet_type = wavelet_type

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        batch_size, channels, height, width = x.shape
        if height % 2 != 0 or width % 2 != 0:
            raise ValueError(f"Wavelet input must have even spatial size, got {x.shape}")
        dec_lo, dec_hi, _, _ = _wavelet_filters(self.wavelet_type)
        base_filters = _stack_filters(dec_lo, dec_hi, x.device, x.dtype)
        padding = max(len(dec_lo) // 2 - 1, 0)
        weight = base_filters.repeat(channels, 1, 1, 1)
        transformed = F.conv2d(x, weight, stride=2, padding=padding, groups=channels)
        transformed = transformed.view(batch_size, channels, 4, transformed.shape[-2], transformed.shape[-1])
        return transformed[:, :, 0], transformed[:, :, 1], transformed[:, :, 2], transformed[:, :, 3]


class InverseWaveletTransform2d(nn.Module):
    def __init__(self, wavelet_type: str = "haar") -> None:
        super().__init__()
        self.wavelet_type = wavelet_type

    def forward(
        self,
        ll: torch.Tensor,
        lh: torch.Tensor,
        hl: torch.Tensor,
        hh: torch.Tensor,
    ) -> torch.Tensor:
        _, channels, _, _ = ll.shape
        _, _, rec_lo, rec_hi = _wavelet_filters(self.wavelet_type)
        base_filters = _stack_filters(rec_lo, rec_hi, ll.device, ll.dtype)
        padding = max(len(rec_lo) // 2 - 1, 0)
        packed = torch.stack([ll, lh, hl, hh], dim=2).reshape(ll.shape[0], channels * 4, ll.shape[-2], ll.shape[-1])
        weight = base_filters.repeat(channels, 1, 1, 1)
        return F.conv_transpose2d(packed, weight, stride=2, padding=padding, groups=channels)


class RawAttentionSkip(nn.Module):
    def __init__(self, channels: int, reduction_ratio: int = 16, use_lfsa: bool = True, use_hfba: bool = True) -> None:
        super().__init__()
        self.channel_attention = LFSA(channels, reduction_ratio) if use_lfsa else nn.Identity()
        if use_hfba:
            self.spatial = nn.Sequential(
                nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=channels, bias=False),
                nn.BatchNorm2d(channels),
                nn.ReLU(inplace=True),
                nn.Conv2d(channels, 1, kernel_size=1),
            )
        else:
            self.spatial = None

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor | None]:
        out = self.channel_attention(x)
        if self.spatial is None:
            return out, None
        boundary_logits = self.spatial(out)
        out = out * torch.sigmoid(boundary_logits)
        return out, boundary_logits


class WBSModule(nn.Module):
    def __init__(
        self,
        channels: int,
        reduction_ratio: int = 16,
        use_wavelet: bool = True,
        use_lfsa: bool = True,
        use_hfba: bool = True,
        boundary_supervision: bool = True,
        wavelet_type: str = "haar",
    ) -> None:
        super().__init__()
        self.use_wavelet = use_wavelet
        self.boundary_supervision = boundary_supervision
        self.dwt = WaveletTransform2d(wavelet_type=wavelet_type)
        self.idwt = InverseWaveletTransform2d(wavelet_type=wavelet_type)
        self.lfsa = LFSA(channels, reduction_ratio) if use_lfsa else nn.Identity()
        self.hfba = HFBA(channels) if use_hfba else None
        self.raw_attention = RawAttentionSkip(channels, reduction_ratio, use_lfsa, use_hfba)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor | None]:
        if not self.use_wavelet:
            refined, boundary_logits = self.raw_attention(x)
            if not self.boundary_supervision:
                boundary_logits = None
            return refined, boundary_logits

        ll, lh, hl, hh = self.dwt(x)
        ll = self.lfsa(ll)
        if self.hfba is not None:
            hf, boundary_logits = self.hfba(lh, hl, hh)
            gate = torch.sigmoid(self.hfba.boundary_head(hf))
            lh = lh * gate
            hl = hl * gate
            hh = hh * gate
        else:
            boundary_logits = None
        refined = self.idwt(ll, lh, hl, hh)
        if not self.boundary_supervision:
            boundary_logits = None
        return refined, boundary_logits

    @torch.no_grad()
    def debug_subbands(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        if not self.use_wavelet:
            refined, boundary_logits = self.raw_attention(x)
            return {
                "refined": refined,
                "boundary_logits": boundary_logits if boundary_logits is not None else x.new_zeros((x.shape[0], 1, x.shape[-2], x.shape[-1])),
            }
        ll, lh, hl, hh = self.dwt(x)
        ll_refined = self.lfsa(ll)
        if self.hfba is not None:
            hf, boundary_logits = self.hfba(lh, hl, hh)
        else:
            hf = torch.zeros_like(ll)
            boundary_logits = torch.zeros((x.shape[0], 1, ll.shape[-2], ll.shape[-1]), device=x.device, dtype=x.dtype)
        return {
            "ll": ll,
            "lh": lh,
            "hl": hl,
            "hh": hh,
            "ll_refined": ll_refined,
            "hf": hf,
            "boundary_logits": boundary_logits,
        }


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_channels: int, skip_channels: int, out_channels: int) -> None:
        super().__init__()
        self.conv = ConvBlock(in_channels + skip_channels, out_channels)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels: int, out_channels: int, stride: int = 1) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.downsample = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out + identity)
        return out


class ResNetEncoder(nn.Module):
    def __init__(self, in_channels: int = 3) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, 64, blocks=3, stride=1)
        self.layer2 = self._make_layer(64, 128, blocks=4, stride=2)
        self.layer3 = self._make_layer(128, 256, blocks=6, stride=2)
        self.layer4 = self._make_layer(256, 512, blocks=3, stride=2)

    @staticmethod
    def _make_layer(in_channels: int, out_channels: int, blocks: int, stride: int) -> nn.Sequential:
        layers = [BasicBlock(in_channels, out_channels, stride=stride)]
        for _ in range(1, blocks):
            layers.append(BasicBlock(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        stem = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(stem)
        layer1 = self.layer1(x)
        layer2 = self.layer2(layer1)
        layer3 = self.layer3(layer2)
        layer4 = self.layer4(layer3)
        return stem, layer1, layer2, layer3, layer4

    def load_imagenet_pretrained(self, fail_silently: bool = True) -> None:
        try:
            from torchvision.models import ResNet34_Weights, resnet34

            state = resnet34(weights=ResNet34_Weights.IMAGENET1K_V1).state_dict()
            # torchvision keys: "conv1.*"/"bn1.*" -> our encoder: "stem.conv1.*"/"stem.bn1.*"
            remapped = {}
            for key, value in state.items():
                if key.startswith("conv1.") or key.startswith("bn1."):
                    remapped["stem." + key] = value
                elif key.startswith("layer"):
                    remapped[key] = value
            current = self.state_dict()
            compatible = {k: v for k, v in remapped.items()
                          if k in current and tuple(v.shape) == tuple(current[k].shape)}
            self.load_state_dict(compatible, strict=False)
            print(f"Loaded {len(compatible)} ImageNet encoder tensors.")
        except Exception as exc:
            message = "ImageNet pretrained weights were unavailable; continuing with random encoder initialization."
            if fail_silently:
                warnings.warn(f"{message} Original error: {exc}")
            else:
                raise RuntimeError(message) from exc


class WBSNet(nn.Module):
    def __init__(self, config: dict[str, Any]) -> None:
        super().__init__()
        decoder_channels = config.get("decoder_channels", [256, 128, 64, 32])
        reduction_ratio = int(config.get("reduction_ratio", 16))
        self.encoder = ResNetEncoder(in_channels=3)
        self.skip_modules = nn.ModuleList(
            [
                WBSModule(
                    64,
                    reduction_ratio=reduction_ratio,
                    use_wavelet=bool(config.get("use_wavelet", True)),
                    use_lfsa=bool(config.get("use_lfsa", True)),
                    use_hfba=bool(config.get("use_hfba", True)),
                    boundary_supervision=bool(config.get("boundary_supervision", True)),
                    wavelet_type=str(config.get("wavelet_type", "haar")),
                ),
                WBSModule(
                    64,
                    reduction_ratio=reduction_ratio,
                    use_wavelet=bool(config.get("use_wavelet", True)),
                    use_lfsa=bool(config.get("use_lfsa", True)),
                    use_hfba=bool(config.get("use_hfba", True)),
                    boundary_supervision=bool(config.get("boundary_supervision", True)),
                    wavelet_type=str(config.get("wavelet_type", "haar")),
                ),
                WBSModule(
                    128,
                    reduction_ratio=reduction_ratio,
                    use_wavelet=bool(config.get("use_wavelet", True)),
                    use_lfsa=bool(config.get("use_lfsa", True)),
                    use_hfba=bool(config.get("use_hfba", True)),
                    boundary_supervision=bool(config.get("boundary_supervision", True)),
                    wavelet_type=str(config.get("wavelet_type", "haar")),
                ),
                WBSModule(
                    256,
                    reduction_ratio=reduction_ratio,
                    use_wavelet=bool(config.get("use_wavelet", True)),
                    use_lfsa=bool(config.get("use_lfsa", True)),
                    use_hfba=bool(config.get("use_hfba", True)),
                    boundary_supervision=bool(config.get("boundary_supervision", True)),
                    wavelet_type=str(config.get("wavelet_type", "haar")),
                ),
            ]
        )
        self.dec4 = DecoderBlock(512, 256, decoder_channels[0])
        self.dec3 = DecoderBlock(decoder_channels[0], 128, decoder_channels[1])
        self.dec2 = DecoderBlock(decoder_channels[1], 64, decoder_channels[2])
        self.dec1 = DecoderBlock(decoder_channels[2], 64, decoder_channels[3])
        self.head = nn.Conv2d(decoder_channels[3], 1, kernel_size=1)

    def forward(self, x: torch.Tensor) -> dict[str, Any]:
        stem, layer1, layer2, layer3, bottleneck = self.encoder(x)
        skip1, bnd1 = self.skip_modules[0](stem)
        skip2, bnd2 = self.skip_modules[1](layer1)
        skip3, bnd3 = self.skip_modules[2](layer2)
        skip4, bnd4 = self.skip_modules[3](layer3)

        dec4 = self.dec4(bottleneck, skip4)
        dec3 = self.dec3(dec4, skip3)
        dec2 = self.dec2(dec3, skip2)
        dec1 = self.dec1(dec2, skip1)
        logits = self.head(F.interpolate(dec1, scale_factor=2, mode="bilinear", align_corners=False))
        boundary_logits = [item for item in [bnd1, bnd2, bnd3, bnd4] if item is not None]
        return {"logits": logits, "boundary_logits": boundary_logits}


def build_experiment_config(
    train_dataset: str,
    variant_id: str,
    seed: int,
    boundary_loss_weight: float = 0.5,
    epochs_override: int | None = None,
    extra_eval_specs: list[dict[str, str]] | None = None,
    run_group: str = "paper_suite",
    run_suffix: str | None = None,
) -> dict[str, Any]:
    config = copy.deepcopy(RESEARCH_CONFIG)
    config["train_dataset"] = train_dataset
    config["variant_id"] = variant_id
    config["variant_display_name"] = VARIANT_SPECS[variant_id]["display_name"]
    config["paper_name"] = VARIANT_SPECS[variant_id]["paper_name"]
    config["seed"] = seed
    config["epochs"] = int(epochs_override or DATASET_SPECS[train_dataset]["epochs"])
    config["boundary_loss_weight"] = float(boundary_loss_weight)
    config.update(VARIANT_SPECS[variant_id]["overrides"])

    eval_specs = [{"dataset": train_dataset, "split": resolve_eval_split(train_dataset)}]
    if extra_eval_specs:
        eval_specs.extend(extra_eval_specs)
    else:
        for item in DATASET_SPECS[train_dataset]["cross_dataset_targets"]:
            if (PROCESSED_ROOT / item["dataset"] / item["split"]).exists():
                eval_specs.append(dict(item))
    config["eval_specs"] = eval_specs

    suffix = f"_{run_suffix}" if run_suffix else ""
    config["run_name"] = f"{train_dataset}_{variant_id}_seed{seed}{suffix}"
    config["run_group"] = run_group
    return config


def count_parameters(model: nn.Module) -> tuple[int, int]:
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    return total, trainable


def prepare_run_dirs(config: dict[str, Any]) -> dict[str, Path]:
    run_dir = OUTPUT_ROOT / config["run_group"] / config["train_dataset"] / config["variant_id"] / f"seed_{config['seed']}"
    if config.get("run_name"):
        run_dir = run_dir / config["run_name"]
    dirs = {
        "run_dir": run_dir,
        "checkpoint_dir": run_dir / "checkpoints",
        "evaluation_dir": run_dir / "evaluation",
        "predictions_dir": run_dir / "predictions",
        "analysis_dir": run_dir / "analysis",
    }
    for path in dirs.values():
        path.mkdir(parents=True, exist_ok=True)
    return dirs


## Training, Evaluation, and Export Utilities

These cells implement:
- the segmentation and boundary losses
- all reported metrics including `HD95`
- checkpointing and run summaries
- prediction export with contact sheets
- per-sample metrics for qualitative review and lesion-size breakdown


In [ ]:
def dice_loss_from_logits(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    dims = tuple(range(1, targets.ndim))
    intersection = (probs * targets).sum(dim=dims)
    union = probs.sum(dim=dims) + targets.sum(dim=dims)
    dice = (2.0 * intersection + eps) / (union + eps)
    return 1.0 - dice.mean()


def segmentation_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    return F.binary_cross_entropy_with_logits(logits, targets) + dice_loss_from_logits(logits, targets)


def boundary_loss(boundary_logits: list[torch.Tensor], boundary_targets: torch.Tensor) -> torch.Tensor:
    if not boundary_logits:
        return boundary_targets.new_tensor(0.0)
    losses = []
    for logits in boundary_logits:
        upsampled_logits = F.interpolate(logits, size=boundary_targets.shape[-2:], mode="bilinear", align_corners=False)
        losses.append(F.binary_cross_entropy_with_logits(upsampled_logits, boundary_targets))
    return torch.stack(losses).mean()


def total_loss(
    model_output: dict[str, Any],
    masks: torch.Tensor,
    boundaries: torch.Tensor,
    boundary_weight: float,
) -> tuple[torch.Tensor, dict[str, float]]:
    seg = segmentation_loss(model_output["logits"], masks)
    bnd = boundary_loss(model_output.get("boundary_logits", []), boundaries)
    total = seg + boundary_weight * bnd
    return total, {
        "segmentation_loss": float(seg.detach().item()),
        "boundary_loss": float(bnd.detach().item()),
        "total_loss": float(total.detach().item()),
    }


def safe_divide(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if denominator > 0 else 0.0


def mask_boundary(mask: np.ndarray) -> np.ndarray:
    padded = np.pad(mask.astype(np.uint8), 1, mode="constant")
    neighbors = [
        padded[1:-1, :-2],
        padded[1:-1, 2:],
        padded[:-2, 1:-1],
        padded[2:, 1:-1],
    ]
    same = np.logical_and.reduce([mask == neighbor for neighbor in neighbors])
    return np.logical_and(mask.astype(bool), np.logical_not(same))


def pairwise_min_distances(points_a: np.ndarray, points_b: np.ndarray, chunk_size: int = 1024) -> np.ndarray:
    if len(points_a) == 0 or len(points_b) == 0:
        return np.array([], dtype=np.float32)
    mins: list[np.ndarray] = []
    for start in range(0, len(points_a), chunk_size):
        chunk = points_a[start : start + chunk_size]
        distances = np.sqrt(((chunk[:, None, :] - points_b[None, :, :]) ** 2).sum(axis=2))
        mins.append(distances.min(axis=1))
    return np.concatenate(mins, axis=0)


def hd95_score(pred_mask: np.ndarray, target_mask: np.ndarray) -> float:
    pred_mask = pred_mask.astype(bool)
    target_mask = target_mask.astype(bool)
    if pred_mask.sum() == 0 and target_mask.sum() == 0:
        return 0.0
    if pred_mask.sum() == 0 or target_mask.sum() == 0:
        return float(np.hypot(*pred_mask.shape))
    pred_boundary = np.argwhere(mask_boundary(pred_mask))
    target_boundary = np.argwhere(mask_boundary(target_mask))
    if len(pred_boundary) == 0 or len(target_boundary) == 0:
        return float(np.hypot(*pred_mask.shape))
    pred_to_target = pairwise_min_distances(pred_boundary, target_boundary)
    target_to_pred = pairwise_min_distances(target_boundary, pred_boundary)
    distances = np.concatenate([pred_to_target, target_to_pred], axis=0)
    return float(np.percentile(distances, 95))


def compute_binary_metrics_from_arrays(pred_mask: np.ndarray, target_mask: np.ndarray) -> dict[str, float]:
    pred_bool = pred_mask.astype(bool)
    target_bool = target_mask.astype(bool)
    tp = float(np.logical_and(pred_bool, target_bool).sum())
    fp = float(np.logical_and(pred_bool, np.logical_not(target_bool)).sum())
    fn = float(np.logical_and(np.logical_not(pred_bool), target_bool).sum())
    tn = float(np.logical_and(np.logical_not(pred_bool), np.logical_not(target_bool)).sum())
    return {
        "dice": safe_divide(2.0 * tp, 2.0 * tp + fp + fn),
        "iou": safe_divide(tp, tp + fp + fn),
        "precision": safe_divide(tp, tp + fp),
        "recall": safe_divide(tp, tp + fn),
        "accuracy": safe_divide(tp + tn, tp + tn + fp + fn),
        "specificity": safe_divide(tn, tn + fp),
        "hd95": hd95_score(pred_bool, target_bool),
        "target_area_ratio": float(target_bool.mean()),
        "pred_area_ratio": float(pred_bool.mean()),
    }


@dataclass
class BinarySegmentationMeter:
    threshold: float = 0.5
    compute_hd95: bool = True
    counts: dict[str, float] = field(
        default_factory=lambda: {
            "tp": 0.0,
            "fp": 0.0,
            "fn": 0.0,
            "tn": 0.0,
            "loss": 0.0,
            "num_batches": 0.0,
        }
    )
    hd95_values: list[float] = field(default_factory=list)
    dice_values: list[float] = field(default_factory=list)
    iou_values: list[float] = field(default_factory=list)

    def update(self, logits: torch.Tensor, targets: torch.Tensor, loss_value: float, training: bool = False) -> None:
        probs = torch.sigmoid(logits)
        preds = (probs >= self.threshold).float()
        self.counts["tp"] += float(((preds == 1) & (targets == 1)).sum().item())
        self.counts["fp"] += float(((preds == 1) & (targets == 0)).sum().item())
        self.counts["fn"] += float(((preds == 0) & (targets == 1)).sum().item())
        self.counts["tn"] += float(((preds == 0) & (targets == 0)).sum().item())
        self.counts["loss"] += float(loss_value)
        self.counts["num_batches"] += 1.0
        # Per-sample Dice/IoU (matches published benchmark conventions)
        pred_np = preds.detach().cpu().numpy()
        target_np = targets.detach().cpu().numpy()
        for pred_s, target_s in zip(pred_np, target_np):
            p, t = pred_s[0].astype(bool), target_s[0].astype(bool)
            tp_s = float(np.logical_and(p, t).sum())
            fp_s = float(np.logical_and(p, ~t).sum())
            fn_s = float(np.logical_and(~p, t).sum())
            self.dice_values.append(safe_divide(2.0 * tp_s, 2.0 * tp_s + fp_s + fn_s))
            self.iou_values.append(safe_divide(tp_s, tp_s + fp_s + fn_s))
        if self.compute_hd95 and not training:
            for pred_s, target_s in zip(pred_np, target_np):
                self.hd95_values.append(hd95_score(pred_s[0], target_s[0]))

    def compute(self) -> dict[str, float]:
        # Use per-sample mean Dice/IoU to match published benchmark conventions (PraNet, Polyp-PVT, SANet)
        dice = float(np.mean(self.dice_values)) if self.dice_values else 0.0
        iou = float(np.mean(self.iou_values)) if self.iou_values else 0.0
        tp = self.counts["tp"]
        fp = self.counts["fp"]
        fn = self.counts["fn"]
        tn = self.counts["tn"]
        metrics = {
            "loss": safe_divide(self.counts["loss"], self.counts["num_batches"]),
            "dice": dice,
            "iou": iou,
            "precision": safe_divide(tp, tp + fp),
            "recall": safe_divide(tp, tp + fn),
            "accuracy": safe_divide(tp + tn, tp + tn + fp + fn),
            "specificity": safe_divide(tn, tn + fp),
        }
        if self.hd95_values:
            metrics["hd95"] = float(np.mean(self.hd95_values))
        return metrics


def mean_scalar_dict(totals: dict[str, float], count: int) -> dict[str, float]:
    if count <= 0:
        return {key: 0.0 for key in totals}
    return {key: float(value / count) for key, value in totals.items()}


def format_metrics(metrics: dict[str, float], ordered_keys: list[str]) -> str:
    return " | ".join(f"{key}: {metrics[key]:.4f}" for key in ordered_keys if key in metrics)


In [ ]:
def make_overlay(image_np: np.ndarray, target_np: np.ndarray, pred_np: np.ndarray) -> np.ndarray:
    overlay = image_np.copy()
    overlay[..., 0] = np.where(pred_np > 0, 255, overlay[..., 0])
    overlay[..., 1] = np.where(target_np > 0, 255, overlay[..., 1])
    return overlay


def tile_with_title(tile: np.ndarray, title: str, title_height: int = 28) -> np.ndarray:
    canvas = np.full((tile.shape[0] + title_height, tile.shape[1], 3), 255, dtype=np.uint8)
    canvas[title_height:, :, :] = tile
    image = Image.fromarray(canvas)
    draw = ImageDraw.Draw(image)
    draw.text((8, 6), title, fill=(0, 0, 0))
    return np.asarray(image)


def find_focus_box(target_np: np.ndarray, pred_np: np.ndarray, min_size: int = 96, padding: int = 24) -> tuple[int, int, int, int]:
    target_mask = target_np > 0
    pred_mask = pred_np > 0
    focus = np.logical_xor(target_mask, pred_mask)
    if not focus.any():
        focus = np.logical_or(target_mask, pred_mask)

    height, width = target_np.shape[:2]
    if not focus.any():
        crop = min(min_size, height, width)
        top = max((height - crop) // 2, 0)
        left = max((width - crop) // 2, 0)
        return left, top, min(left + crop, width), min(top + crop, height)

    ys, xs = np.where(focus)
    left = max(int(xs.min()) - padding, 0)
    right = min(int(xs.max()) + padding + 1, width)
    top = max(int(ys.min()) - padding, 0)
    bottom = min(int(ys.max()) + padding + 1, height)

    box_width = right - left
    box_height = bottom - top
    target_size = max(min_size, box_width, box_height)
    center_x = (left + right) // 2
    center_y = (top + bottom) // 2
    half = target_size // 2
    left = max(center_x - half, 0)
    top = max(center_y - half, 0)
    right = min(left + target_size, width)
    bottom = min(top + target_size, height)
    left = max(right - target_size, 0)
    top = max(bottom - target_size, 0)
    return left, top, right, bottom


def draw_focus_box(tile: np.ndarray, box: tuple[int, int, int, int], color: tuple[int, int, int] = (255, 0, 0)) -> np.ndarray:
    image = Image.fromarray(tile)
    draw = ImageDraw.Draw(image)
    left, top, right, bottom = box
    draw.rectangle([left, top, right - 1, bottom - 1], outline=color, width=3)
    return np.asarray(image)


def create_prediction_visuals(
    image: torch.Tensor,
    target_mask: torch.Tensor,
    pred_mask: torch.Tensor,
    mean: list[float],
    std: list[float],
    zoom_size: int = 256,
) -> dict[str, np.ndarray | tuple[int, int, int, int]]:
    image_np = denormalize_image_tensor(image, mean, std)
    target_np = tensor_to_mask_uint8(target_mask)
    pred_np = tensor_to_mask_uint8(pred_mask)
    overlay_np = make_overlay(image_np, target_np, pred_np)
    focus_box = find_focus_box(target_np, pred_np)
    left, top, right, bottom = focus_box
    image_box = draw_focus_box(image_np, focus_box)
    target_box = draw_focus_box(np.repeat(target_np[..., None], 3, axis=2), focus_box)
    pred_box = draw_focus_box(np.repeat(pred_np[..., None], 3, axis=2), focus_box)
    overlay_box = draw_focus_box(overlay_np, focus_box)
    zoom_crop = overlay_np[top:bottom, left:right]
    if zoom_crop.size == 0:
        zoom_crop = overlay_np
    base_height, base_width = image_box.shape[:2]
    zoom_render_size = max(int(zoom_size), base_height, base_width)
    zoom_panel = np.asarray(
        Image.fromarray(zoom_crop)
        .resize((zoom_render_size, zoom_render_size), resample=RESAMPLING.NEAREST)
        .resize((base_width, base_height), resample=RESAMPLING.NEAREST)
    )
    paper_panel = np.concatenate(
        [
            tile_with_title(image_box, "Image"),
            tile_with_title(target_box, "Ground Truth"),
            tile_with_title(pred_box, "Prediction"),
            tile_with_title(overlay_box, "Overlay"),
            tile_with_title(zoom_panel, "Boundary Zoom"),
        ],
        axis=1,
    )
    return {
        "image": image_np,
        "target": target_np,
        "prediction": pred_np,
        "overlay": overlay_np,
        "focus_box": focus_box,
        "paper_panel": paper_panel,
    }


def save_prediction_triplet(
    save_dir: str | Path,
    sample_id: str,
    image: torch.Tensor,
    target_mask: torch.Tensor,
    pred_mask: torch.Tensor,
    mean: list[float],
    std: list[float],
) -> dict[str, str]:
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    visuals = create_prediction_visuals(image, target_mask, pred_mask, mean, std)
    image_path = save_dir / f"{sample_id}_image.png"
    target_path = save_dir / f"{sample_id}_target.png"
    prediction_path = save_dir / f"{sample_id}_prediction.png"
    overlay_path = save_dir / f"{sample_id}_overlay.png"
    panel_path = save_dir / f"{sample_id}_paper_panel.png"
    Image.fromarray(visuals["image"]).save(image_path)
    Image.fromarray(visuals["target"]).save(target_path)
    Image.fromarray(visuals["prediction"]).save(prediction_path)
    Image.fromarray(visuals["overlay"]).save(overlay_path)
    Image.fromarray(visuals["paper_panel"]).save(panel_path)
    return {
        "image": str(image_path),
        "target": str(target_path),
        "prediction": str(prediction_path),
        "overlay": str(overlay_path),
        "paper_panel": str(panel_path),
    }


def save_contact_sheet(
    panel_paths: list[str | Path],
    output_path: str | Path,
    columns: int = 2,
    panel_spacing: int = 20,
    margin: int = 20,
) -> str:
    if not panel_paths:
        raise ValueError("At least one panel is required for a contact sheet.")
    opened_panels = [Image.open(path).convert("RGB") for path in panel_paths]
    try:
        max_width = max(panel.width for panel in opened_panels)
        max_height = max(panel.height for panel in opened_panels)
        columns = max(1, columns)
        rows = (len(opened_panels) + columns - 1) // columns
        canvas_width = (columns * max_width) + ((columns - 1) * panel_spacing) + (2 * margin)
        canvas_height = (rows * max_height) + ((rows - 1) * panel_spacing) + (2 * margin)
        canvas = Image.new("RGB", (canvas_width, canvas_height), (255, 255, 255))
        for index, panel in enumerate(opened_panels):
            row = index // columns
            column = index % columns
            x = margin + column * (max_width + panel_spacing)
            y = margin + row * (max_height + panel_spacing)
            canvas.paste(panel, (x, y))
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        canvas.save(output_path)
        return str(output_path)
    finally:
        for panel in opened_panels:
            panel.close()


def build_optimizer(model: nn.Module, config: dict[str, Any]) -> tuple[torch.optim.Optimizer, torch.optim.lr_scheduler._LRScheduler]:
    encoder_params = list(model.encoder.parameters())
    encoder_param_ids = {id(param) for param in encoder_params}
    decoder_params = [param for param in model.parameters() if id(param) not in encoder_param_ids]
    optimizer = torch.optim.AdamW(
        [
            {"params": encoder_params, "lr": float(config["encoder_lr"])},
            {"params": decoder_params, "lr": float(config["decoder_lr"])},
        ],
        weight_decay=float(config["weight_decay"]),
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=int(config["epochs"]))
    return optimizer, scheduler


def save_checkpoint(
    path: str | Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler._LRScheduler,
    scaler: Any,
    epoch: int,
    best_metric: float,
    config: dict[str, Any],
) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "epoch": epoch,
            "best_metric": best_metric,
            "config": config,
            "state_dict": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
        },
        path,
    )


def load_checkpoint(
    path: str | Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer | None = None,
    scheduler: torch.optim.lr_scheduler._LRScheduler | None = None,
    scaler: Any | None = None,
    map_location: str | torch.device = "cpu",
) -> dict[str, Any]:
    checkpoint = torch.load(path, map_location=map_location)
    model.load_state_dict(checkpoint["state_dict"], strict=True)
    if optimizer is not None and "optimizer" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer"])
    if scheduler is not None and "scheduler" in checkpoint:
        scheduler.load_state_dict(checkpoint["scheduler"])
    if scaler is not None and "scaler" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler"])
    return checkpoint


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer | None,
    scaler: Any,
    device: torch.device,
    config: dict[str, Any],
    training: bool,
) -> dict[str, float]:
    amp_enabled = bool(config["amp"] and device.type == "cuda")
    grad_accum_steps = int(config["grad_accum_steps"])
    clip_grad_norm = float(config["clip_grad_norm"])
    meter = BinarySegmentationMeter(
        threshold=float(config["threshold"]),
        compute_hd95=bool(config["compute_hd95"]),
    )
    loss_totals = {"segmentation_loss": 0.0, "boundary_loss": 0.0, "total_loss": 0.0}
    loss_steps = 0

    model.train(training)
    if training and optimizer is not None:
        optimizer.zero_grad(set_to_none=True)

    progress = tqdm(loader, desc="train" if training else "eval", leave=False)
    for step, batch in enumerate(progress, start=1):
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)
        boundaries = batch["boundary"].to(device, non_blocking=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=device.type, enabled=amp_enabled):
                output = model(images)
                loss, loss_parts = total_loss(
                    output,
                    masks,
                    boundaries,
                    boundary_weight=float(config["boundary_loss_weight"]),
                )
                scaled_loss = loss / grad_accum_steps

            if training and optimizer is not None:
                scaler.scale(scaled_loss).backward()
                if step % grad_accum_steps == 0:
                    if clip_grad_norm > 0:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad_norm)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

        meter.update(output["logits"].detach(), masks.detach(), float(loss.detach().item()))
        for key in loss_totals:
            loss_totals[key] += float(loss_parts[key])
        loss_steps += 1
        running_metrics = meter.compute()
        progress.set_postfix(loss=f"{loss.item():.4f}", dice=f"{running_metrics.get('dice', 0.0):.4f}")

    if training and optimizer is not None and len(loader) % grad_accum_steps != 0:
        if clip_grad_norm > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad_norm)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    metrics = meter.compute()
    metrics.update(mean_scalar_dict(loss_totals, loss_steps))
    return metrics


@torch.no_grad()
def evaluate_and_export(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    config: dict[str, Any],
    save_dir: str | Path | None = None,
) -> tuple[dict[str, float], pd.DataFrame]:
    model.eval()
    meter = BinarySegmentationMeter(
        threshold=float(config["threshold"]),
        compute_hd95=bool(config["compute_hd95"]),
    )
    loss_totals = {"segmentation_loss": 0.0, "boundary_loss": 0.0, "total_loss": 0.0}
    loss_steps = 0
    sample_records: list[dict[str, Any]] = []
    saved_count = 0
    paper_panel_paths: list[str] = []
    export_errors: list[dict[str, str]] = []

    progress = tqdm(loader, desc="predict", leave=False)
    for batch in progress:
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)
        boundaries = batch["boundary"].to(device, non_blocking=True)
        output = model(images)
        loss, loss_parts = total_loss(output, masks, boundaries, float(config["boundary_loss_weight"]))
        probs = (torch.sigmoid(output["logits"]) >= float(config["threshold"])).float()
        meter.update(output["logits"], masks, float(loss.item()))
        for key in loss_totals:
            loss_totals[key] += float(loss_parts[key])
        loss_steps += 1

        pred_np = probs.detach().cpu().numpy()
        target_np = masks.detach().cpu().numpy()
        for idx in range(images.shape[0]):
            sample_id = str(batch["sample_id"][idx])
            metrics = compute_binary_metrics_from_arrays(pred_np[idx, 0], target_np[idx, 0])
            sample_records.append(
                {
                    "sample_id": sample_id,
                    **metrics,
                    "image_path": batch["image_path"][idx],
                    "mask_path": batch["mask_path"][idx],
                    "boundary_path": batch["boundary_path"][idx],
                }
            )
            if save_dir is not None and saved_count < int(config["max_visualizations"]):
                try:
                    saved_paths = save_prediction_triplet(
                        save_dir=save_dir,
                        sample_id=sample_id,
                        image=batch["image"][idx].cpu(),
                        target_mask=batch["mask"][idx].cpu(),
                        pred_mask=probs[idx].cpu(),
                        mean=config["normalize_mean"],
                        std=config["normalize_std"],
                    )
                except Exception as exc:
                    if bool(config.get("tolerate_export_failures", True)):
                        warnings.warn(f"Skipping visualization export for {sample_id}: {exc}")
                        export_errors.append({"sample_id": sample_id, "error": repr(exc)})
                    else:
                        raise
                else:
                    paper_panel_paths.append(saved_paths["paper_panel"])
                    saved_count += 1

    if save_dir is not None and paper_panel_paths:
        try:
            save_contact_sheet(
                panel_paths=paper_panel_paths,
                output_path=Path(save_dir) / "paper_contact_sheet.png",
                columns=int(config["contact_sheet_columns"]),
            )
        except Exception as exc:
            if bool(config.get("tolerate_export_failures", True)):
                warnings.warn(f"Skipping contact sheet export because: {exc}")
                export_errors.append({"sample_id": "contact_sheet", "error": repr(exc)})
            else:
                raise
    if save_dir is not None and export_errors:
        save_json(Path(save_dir).parent / "export_errors.json", export_errors)

    metrics = meter.compute()
    metrics.update(mean_scalar_dict(loss_totals, loss_steps))
    sample_frame = pd.DataFrame(sample_records)
    return metrics, sample_frame


def instantiate_training_components(config: dict[str, Any]) -> dict[str, Any]:
    train_dataset = ProcessedWBSNetDataset(
        processed_root=PROCESSED_ROOT,
        dataset_name=config["train_dataset"],
        split="train",
        train=True,
        config=config,
    )
    val_dataset = ProcessedWBSNetDataset(
        processed_root=PROCESSED_ROOT,
        dataset_name=config["train_dataset"],
        split="val",
        train=False,
        config=config,
    )
    train_loader = build_dataloader(train_dataset, batch_size=config["batch_size"], shuffle=True, config=config)
    val_loader = build_dataloader(val_dataset, batch_size=config["batch_size"], shuffle=False, config=config)

    model = WBSNet(config).to(DEVICE)
    if bool(config.get("encoder_pretrained", True)):
        model.encoder.load_imagenet_pretrained(fail_silently=bool(config.get("allow_pretrained_fallback", True)))
    if bool(config.get("compile_model", False)) and hasattr(torch, "compile"):
        model = torch.compile(model)
    optimizer, scheduler = build_optimizer(model, config)
    scaler = create_grad_scaler(DEVICE, enabled=bool(config["amp"]))
    return {
        "model": model,
        "optimizer": optimizer,
        "scheduler": scheduler,
        "scaler": scaler,
        "train_loader": train_loader,
        "val_loader": val_loader,
        "train_dataset": train_dataset,
        "val_dataset": val_dataset,
    }


def load_history_rows(history_path: str | Path) -> list[dict[str, Any]]:
    history_path = Path(history_path)
    if not history_path.exists():
        return []
    try:
        history_df = pd.read_csv(history_path)
    except Exception as exc:
        warnings.warn(f"Could not read training history from {history_path}: {exc}")
        return []
    if history_df.empty:
        return []
    return history_df.to_dict(orient="records")


def build_run_summary(
    config: dict[str, Any],
    best_checkpoint_path: Path,
    best_metrics: dict[str, Any],
    total_params: int,
    trainable_params: int,
    evaluation_payloads: list[dict[str, Any]],
) -> dict[str, Any]:
    return {
        "train_dataset": config["train_dataset"],
        "variant_id": config["variant_id"],
        "variant_display_name": config["variant_display_name"],
        "paper_name": config["paper_name"],
        "seed": config["seed"],
        "run_name": config["run_name"],
        "best_val_metrics": best_metrics,
        "best_checkpoint": str(best_checkpoint_path.resolve()),
        "params_total": total_params,
        "params_trainable": trainable_params,
        "evaluations": evaluation_payloads,
    }


def run_experiment(config: dict[str, Any], skip_existing: bool = True) -> dict[str, Any]:
    seed_everything(int(config["seed"]), deterministic=bool(config["deterministic"]))
    dirs = prepare_run_dirs(config)
    run_summary_path = dirs["run_dir"] / "run_summary.json"
    history_path = dirs["run_dir"] / "metrics.csv"
    best_metrics_path = dirs["run_dir"] / "best_metrics.json"
    best_checkpoint_path = dirs["checkpoint_dir"] / "best.pt"
    last_checkpoint_path = dirs["checkpoint_dir"] / "last.pt"
    if skip_existing and run_summary_path.exists():
        print(f"Skipping existing run: {dirs['run_dir']}")
        return load_json(run_summary_path)

    components = instantiate_training_components(config)
    model = components["model"]
    optimizer = components["optimizer"]
    scheduler = components["scheduler"]
    scaler = components["scaler"]
    train_loader = components["train_loader"]
    val_loader = components["val_loader"]

    total_params, trainable_params = count_parameters(model)
    history_rows = load_history_rows(history_path)
    best_metrics: dict[str, Any] = load_json(best_metrics_path) if best_metrics_path.exists() else {}
    best_metric = float(best_metrics.get("dice", float("-inf"))) if best_metrics else float("-inf")
    start_epoch = 0

    save_json(dirs["run_dir"] / "resolved_config.json", config)

    if bool(config.get("resume_training", True)) and last_checkpoint_path.exists() and history_rows:
        checkpoint = load_checkpoint(last_checkpoint_path, model, optimizer, scheduler, scaler, map_location=DEVICE)
        start_epoch = int(checkpoint.get("epoch", len(history_rows) - 1)) + 1
        best_metric = float(checkpoint.get("best_metric", best_metric))
        history_rows = [row for row in history_rows if int(row.get("epoch", 0)) <= start_epoch]
        if start_epoch < int(config["epochs"]):
            print(f"Resuming training from epoch {start_epoch + 1}/{int(config['epochs'])} using {last_checkpoint_path.name}.")
        else:
            print(f"Training already complete for {config['run_name']}; reusing checkpoints for evaluation.")

    for epoch in range(start_epoch, int(config["epochs"])):
        train_metrics = run_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scaler=scaler,
            device=DEVICE,
            config=config,
            training=True,
        )
        val_metrics = run_epoch(
            model=model,
            loader=val_loader,
            optimizer=None,
            scaler=scaler,
            device=DEVICE,
            config=config,
            training=False,
        )
        scheduler.step()

        row = {
            "epoch": epoch + 1,
            **{f"train_{key}": value for key, value in train_metrics.items()},
            **{f"val_{key}": value for key, value in val_metrics.items()},
            "encoder_lr": optimizer.param_groups[0]["lr"],
            "decoder_lr": optimizer.param_groups[1]["lr"],
        }
        history_rows.append(row)
        history_df = pd.DataFrame(history_rows)
        history_df.to_csv(history_path, index=False)

        current_metric = float(val_metrics["dice"])
        if current_metric > best_metric:
            best_metric = current_metric
            best_metrics = dict(val_metrics)
            save_checkpoint(
                best_checkpoint_path,
                model,
                optimizer,
                scheduler,
                scaler,
                epoch=epoch,
                best_metric=best_metric,
                config=config,
            )
            save_json(best_metrics_path, best_metrics)

        save_checkpoint(
            last_checkpoint_path,
            model,
            optimizer,
            scheduler,
            scaler,
            epoch=epoch,
            best_metric=best_metric,
            config=config,
        )

        save_every = max(1, min(10, int(config["epochs"])))
        if (epoch + 1) % save_every == 0:
            save_checkpoint(
                dirs["checkpoint_dir"] / f"epoch_{epoch + 1:03d}.pt",
                model,
                optimizer,
                scheduler,
                scaler,
                epoch=epoch,
                best_metric=best_metric,
                config=config,
            )

        print(
            f"{config['run_name']} | epoch {epoch + 1:03d}/{int(config['epochs']):03d} | "
            f"train -> {format_metrics(train_metrics, ['total_loss', 'dice', 'iou', 'hd95'])} | "
            f"val -> {format_metrics(val_metrics, ['total_loss', 'dice', 'iou', 'hd95'])}"
        )

    checkpoint_to_evaluate = best_checkpoint_path if best_checkpoint_path.exists() else last_checkpoint_path
    if not checkpoint_to_evaluate.exists():
        raise FileNotFoundError(f"No checkpoint available for evaluation in {dirs['checkpoint_dir']}")

    load_checkpoint(checkpoint_to_evaluate, model, map_location=DEVICE)
    evaluation_payloads: list[dict[str, Any]] = []
    for eval_spec in config["eval_specs"]:
        eval_dataset_name = eval_spec["dataset"]
        eval_split = eval_spec["split"]
        export_dir = dirs["evaluation_dir"] / f"{eval_dataset_name}_{eval_split}"
        metrics_path = export_dir / "metrics.json"
        if bool(config.get("resume_evaluation", True)) and metrics_path.exists():
            print(f"Reusing existing evaluation: {eval_dataset_name}/{eval_split}")
            evaluation_payloads.append(load_json(metrics_path))
            continue

        eval_dataset = ProcessedWBSNetDataset(
            processed_root=PROCESSED_ROOT,
            dataset_name=eval_dataset_name,
            split=eval_split,
            train=False,
            config=config,
        )
        eval_loader = build_dataloader(eval_dataset, batch_size=config["batch_size"], shuffle=False, config=config)
        metrics, sample_frame = evaluate_and_export(
            model=model,
            loader=eval_loader,
            device=DEVICE,
            config=config,
            save_dir=export_dir / "predictions",
        )
        sample_frame.to_csv(export_dir / "sample_metrics.csv", index=False)
        payload = {
            "train_dataset": config["train_dataset"],
            "eval_dataset": eval_dataset_name,
            "split": eval_split,
            "variant_id": config["variant_id"],
            "paper_name": config["paper_name"],
            "seed": config["seed"],
            "run_name": config["run_name"],
            "checkpoint": str(checkpoint_to_evaluate.resolve()),
            "metrics": metrics,
        }
        save_json(metrics_path, payload)
        evaluation_payloads.append(payload)

    summary = build_run_summary(
        config=config,
        best_checkpoint_path=checkpoint_to_evaluate,
        best_metrics=best_metrics,
        total_params=total_params,
        trainable_params=trainable_params,
        evaluation_payloads=evaluation_payloads,
    )
    save_json(run_summary_path, summary)
    return summary


## Single-Experiment Sanity Check

Use this section first. It builds the configured dataset and model, previews samples, and optionally runs one complete experiment.


In [ ]:
SINGLE_CONFIG = build_experiment_config(
    train_dataset=RESEARCH_CONFIG["single_run_dataset"],
    variant_id=RESEARCH_CONFIG["single_run_variant"],
    seed=int(RESEARCH_CONFIG["single_run_seed"]),
    boundary_loss_weight=0.5,
    epochs_override=RESEARCH_CONFIG["single_run_epochs_override"],
    run_group="single_run",
)

print(json.dumps({key: value for key, value in SINGLE_CONFIG.items() if key not in {'augment', 'eval_specs'}}, indent=2))
print("Evaluation specs:", SINGLE_CONFIG["eval_specs"])

preview_components = instantiate_training_components(SINGLE_CONFIG)
print(f"Train samples: {len(preview_components['train_dataset'])}")
print(f"Val samples:   {len(preview_components['val_dataset'])}")
total_params, trainable_params = count_parameters(preview_components["model"])
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

example_batch = next(iter(preview_components["val_loader"]))
with torch.no_grad():
    example_output = preview_components["model"](example_batch["image"][:2].to(DEVICE))
print(f"Input batch: {tuple(example_batch['image'][:2].shape)}")
print(f"Logits:      {tuple(example_output['logits'].shape)}")
print(f"Boundary heads: {[tuple(item.shape) for item in example_output['boundary_logits']]}")

show_samples(preview_components["val_dataset"], count=3, title=f"{SINGLE_CONFIG['train_dataset']} validation")


In [ ]:
RUN_SINGLE_EXPERIMENT = False

if RUN_SINGLE_EXPERIMENT:
    single_run_summary = run_experiment(SINGLE_CONFIG, skip_existing=True)
    display(pd.DataFrame([single_run_summary]))
else:
    print("Set RUN_SINGLE_EXPERIMENT = True to train or resume the configured model.")


## Full Paper Suite

This section launches the multi-seed paper experiments:
- `A1` to `A7`
- datasets in `paper_suite_datasets`
- seeds in `paper_suite_seeds`

This is intentionally guarded because it is the longest-running part of the notebook.


In [ ]:
def run_paper_suite(
    datasets: list[str],
    variant_ids: list[str],
    seeds: list[int],
    skip_existing: bool = True,
) -> pd.DataFrame:
    records: list[dict[str, Any]] = []
    for dataset_name in datasets:
        for variant_id in variant_ids:
            for seed in seeds:
                config = build_experiment_config(
                    train_dataset=dataset_name,
                    variant_id=variant_id,
                    seed=int(seed),
                    boundary_loss_weight=0.5,
                    run_group="paper_suite",
                )
                summary = run_experiment(config, skip_existing=skip_existing)
                records.append(
                    {
                        "train_dataset": summary["train_dataset"],
                        "variant_id": summary["variant_id"],
                        "seed": summary["seed"],
                        "paper_name": summary["paper_name"],
                        "best_val_dice": summary["best_val_metrics"].get("dice"),
                        "best_checkpoint": summary["best_checkpoint"],
                    }
                )
    return pd.DataFrame(records)


RUN_FULL_PAPER_SUITE = False

if RUN_FULL_PAPER_SUITE:
    suite_frame = run_paper_suite(
        datasets=RESEARCH_CONFIG["paper_suite_datasets"],
        variant_ids=RESEARCH_CONFIG["paper_suite_variants"],
        seeds=RESEARCH_CONFIG["paper_suite_seeds"],
        skip_existing=True,
    )
    display(suite_frame)
else:
    print("Set RUN_FULL_PAPER_SUITE = True to launch the full multi-seed ablation suite.")


## Lambda Sweep

The paper also needs a boundary-loss sensitivity analysis. The next cell sweeps `lambda` over the values in `lambda_sweep_values` using the full model `A2`.


In [ ]:
def run_lambda_sweep(
    train_dataset: str = "kvasir",
    seed: int = 3407,
    lambda_values: list[float] | None = None,
    skip_existing: bool = True,
) -> pd.DataFrame:
    lambda_values = lambda_values or list(RESEARCH_CONFIG["lambda_sweep_values"])
    records: list[dict[str, Any]] = []
    for lambda_value in lambda_values:
        config = build_experiment_config(
            train_dataset=train_dataset,
            variant_id="A2",
            seed=seed,
            boundary_loss_weight=float(lambda_value),
            run_group="lambda_sweep",
            run_suffix=f"lambda_{str(lambda_value).replace('.', '_')}",
        )
        summary = run_experiment(config, skip_existing=skip_existing)
        primary_eval = summary["evaluations"][0]["metrics"] if summary["evaluations"] else {}
        records.append(
            {
                "lambda": lambda_value,
                "train_dataset": train_dataset,
                "seed": seed,
                "dice": primary_eval.get("dice"),
                "iou": primary_eval.get("iou"),
                "hd95": primary_eval.get("hd95"),
            }
        )
    return pd.DataFrame(records).sort_values("lambda").reset_index(drop=True)


RUN_LAMBDA_SWEEP = False

if RUN_LAMBDA_SWEEP:
    lambda_frame = run_lambda_sweep(train_dataset="kvasir", seed=3407)
    display(lambda_frame)

    fig, ax1 = plt.subplots(figsize=(8, 4.5))
    ax2 = ax1.twinx()
    ax1.plot(lambda_frame["lambda"], lambda_frame["dice"], marker="o", color="tab:blue", label="Dice")
    ax2.plot(lambda_frame["lambda"], lambda_frame["hd95"], marker="s", color="tab:red", label="HD95")
    ax1.set_xlabel("Boundary loss weight lambda")
    ax1.set_ylabel("Dice", color="tab:blue")
    ax2.set_ylabel("HD95", color="tab:red")
    ax1.grid(True, alpha=0.25)
    plt.title("Lambda sensitivity sweep")
    plt.tight_layout()
    plt.show()
else:
    print("Set RUN_LAMBDA_SWEEP = True to launch the lambda sweep.")


## Aggregation, Significance, and Paper Tables

The next cell scans the experiment folders produced by the notebook, aggregates metrics across seeds, runs significance tests, and creates paper-style comparison tables.


In [ ]:
LOWER_IS_BETTER = {"hd95", "loss", "segmentation_loss", "boundary_loss", "total_loss"}


def collect_local_records(root: str | Path = OUTPUT_ROOT) -> pd.DataFrame:
    root = Path(root)
    records: list[dict[str, Any]] = []
    for path in root.rglob("run_summary.json"):
        payload = load_json(path)
        records.append(
            {
                "record_type": "run_summary",
                "source": str(path),
                "train_dataset": payload.get("train_dataset"),
                "eval_dataset": payload.get("train_dataset"),
                "split": "val",
                "variant_id": payload.get("variant_id"),
                "paper_name": payload.get("paper_name"),
                "seed": payload.get("seed"),
                **payload.get("best_val_metrics", {}),
            }
        )

    for path in root.rglob("evaluation/*/metrics.json"):
        payload = load_json(path)
        records.append(
            {
                "record_type": "evaluation",
                "source": str(path),
                "train_dataset": payload.get("train_dataset"),
                "eval_dataset": payload.get("eval_dataset"),
                "split": payload.get("split"),
                "variant_id": payload.get("variant_id"),
                "paper_name": payload.get("paper_name"),
                "seed": payload.get("seed"),
                **payload.get("metrics", {}),
            }
        )
    return pd.DataFrame(records)


def aggregate_local_results(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()
    id_columns = {"record_type", "source", "train_dataset", "eval_dataset", "split", "variant_id", "paper_name", "seed"}
    metric_columns = [column for column in frame.columns if column not in id_columns and pd.api.types.is_numeric_dtype(frame[column])]
    grouped = frame.groupby(["record_type", "train_dataset", "eval_dataset", "split", "variant_id", "paper_name"], dropna=False)[metric_columns]
    mean_df = grouped.mean().add_suffix("_mean")
    std_df = grouped.std(ddof=0).fillna(0.0).add_suffix("_std")
    count_df = grouped.size().rename("num_runs")
    return pd.concat([mean_df, std_df, count_df], axis=1).reset_index()


def format_mean_std(mean_value: float | None, std_value: float | None, precision: int = 3) -> str:
    if mean_value is None or pd.isna(mean_value):
        return "---"
    if std_value is None or pd.isna(std_value):
        return f"{mean_value:.{precision}f}"
    return f"{mean_value:.{precision}f} ± {std_value:.{precision}f}"


def paper_main_registry_frame() -> pd.DataFrame:
    records: list[dict[str, Any]] = []
    for dataset_name, methods in PAPER_REPORTED_MAIN.items():
        for method_name, metrics in methods.items():
            records.append(
                {
                    "dataset": dataset_name,
                    "method": method_name,
                    **metrics,
                }
            )
    return pd.DataFrame(records)


def paper_cross_registry_frame() -> pd.DataFrame:
    records: list[dict[str, Any]] = []
    for protocol_name, methods in PAPER_REPORTED_CROSS_DATASET.items():
        for method_name, metrics in methods.items():
            records.append({"protocol": protocol_name, "method": method_name, **metrics})
    return pd.DataFrame(records)


def local_method_name(variant_id: str) -> str:
    if variant_id == "A1":
        return "U-Net (ResNet-34)"
    if variant_id == "A2":
        return "WBSNet (A2)"
    return VARIANT_SPECS[variant_id]["paper_name"]


def build_local_paper_rows(summary_frame: pd.DataFrame) -> pd.DataFrame:
    if summary_frame.empty:
        return pd.DataFrame()
    rows: list[dict[str, Any]] = []
    eval_frame = summary_frame[summary_frame["record_type"] == "evaluation"].copy()
    for _, row in eval_frame.iterrows():
        rows.append(
            {
                "dataset": row["eval_dataset"],
                "train_dataset": row["train_dataset"],
                "split": row["split"],
                "variant_id": row["variant_id"],
                "method": local_method_name(row["variant_id"]),
                "dice_mean": row.get("dice_mean"),
                "dice_std": row.get("dice_std"),
                "iou_mean": row.get("iou_mean"),
                "iou_std": row.get("iou_std"),
                "hd95_mean": row.get("hd95_mean"),
                "hd95_std": row.get("hd95_std"),
                "num_runs": row.get("num_runs"),
            }
        )
    return pd.DataFrame(rows)


def build_combined_main_table(local_summary: pd.DataFrame) -> pd.DataFrame:
    combined = paper_main_registry_frame()
    local_rows = build_local_paper_rows(local_summary)
    if not local_rows.empty:
        local_rows = local_rows[local_rows["dataset"].isin(["kvasir", "cvc_clinicdb", "isic2018"])]
        local_rows = local_rows[local_rows["train_dataset"] == local_rows["dataset"]]
        for _, row in local_rows.iterrows():
            mask = (combined["dataset"] == row["dataset"]) & (combined["method"] == row["method"])
            if mask.any():
                combined.loc[mask, ["dice_mean", "dice_std", "iou_mean", "iou_std", "hd95_mean", "hd95_std"]] = [
                    row["dice_mean"],
                    row["dice_std"],
                    row["iou_mean"],
                    row["iou_std"],
                    row["hd95_mean"],
                    row["hd95_std"],
                ]
            else:
                combined = pd.concat(
                    [
                        combined,
                        pd.DataFrame(
                            [
                                {
                                    "dataset": row["dataset"],
                                    "method": row["method"],
                                    "dice_mean": row["dice_mean"],
                                    "dice_std": row["dice_std"],
                                    "iou_mean": row["iou_mean"],
                                    "iou_std": row["iou_std"],
                                    "hd95_mean": row["hd95_mean"],
                                    "hd95_std": row["hd95_std"],
                                }
                            ]
                        ),
                    ],
                    ignore_index=True,
                )

    display_frame = combined.copy()
    display_frame["Dice"] = [format_mean_std(m, s) for m, s in zip(display_frame["dice_mean"], display_frame["dice_std"])]
    display_frame["IoU"] = [format_mean_std(m, s) for m, s in zip(display_frame["iou_mean"], display_frame["iou_std"])]
    display_frame["HD95"] = [format_mean_std(m, s, precision=1) for m, s in zip(display_frame["hd95_mean"], display_frame["hd95_std"])]
    display_frame = display_frame[["dataset", "method", "Dice", "IoU", "HD95"]]
    return display_frame.sort_values(["dataset", "method"]).reset_index(drop=True)


def build_cross_dataset_table(local_summary: pd.DataFrame) -> pd.DataFrame:
    combined = paper_cross_registry_frame()
    local_rows = build_local_paper_rows(local_summary)
    if not local_rows.empty:
        local_rows = local_rows[
            (local_rows["train_dataset"] == "kvasir")
            & (local_rows["dataset"] == "cvc_colondb")
        ]
        for _, row in local_rows.iterrows():
            method_name = row["method"]
            mask = (combined["protocol"] == "kvasir_to_cvc_colondb") & (combined["method"] == method_name)
            if mask.any():
                combined.loc[mask, ["dice_mean", "dice_std", "iou_mean", "iou_std", "hd95_mean", "hd95_std"]] = [
                    row["dice_mean"],
                    row["dice_std"],
                    row["iou_mean"],
                    row["iou_std"],
                    row["hd95_mean"],
                    row["hd95_std"],
                ]
            elif method_name in {"U-Net (ResNet-34)", "WBSNet (A2)"}:
                combined = pd.concat(
                    [
                        combined,
                        pd.DataFrame(
                            [
                                {
                                    "protocol": "kvasir_to_cvc_colondb",
                                    "method": method_name,
                                    "dice_mean": row["dice_mean"],
                                    "dice_std": row["dice_std"],
                                    "iou_mean": row["iou_mean"],
                                    "iou_std": row["iou_std"],
                                    "hd95_mean": row["hd95_mean"],
                                    "hd95_std": row["hd95_std"],
                                }
                            ]
                        ),
                    ],
                    ignore_index=True,
                )

    display_frame = combined.copy()
    display_frame["Dice"] = [format_mean_std(m, s) for m, s in zip(display_frame["dice_mean"], display_frame["dice_std"])]
    display_frame["IoU"] = [format_mean_std(m, s) for m, s in zip(display_frame["iou_mean"], display_frame["iou_std"])]
    display_frame["HD95"] = [format_mean_std(m, s, precision=1) for m, s in zip(display_frame["hd95_mean"], display_frame["hd95_std"])]
    return display_frame[["protocol", "method", "Dice", "IoU", "HD95"]].sort_values("method").reset_index(drop=True)


def build_ablation_table(local_summary: pd.DataFrame, dataset_name: str = "kvasir") -> pd.DataFrame:
    local_rows = build_local_paper_rows(local_summary)
    if local_rows.empty:
        return pd.DataFrame()
    frame = local_rows[(local_rows["train_dataset"] == dataset_name) & (local_rows["dataset"] == dataset_name)].copy()
    frame = frame[frame["variant_id"].isin(["A1", "A2", "A3", "A4", "A5", "A6", "A7"])]
    if frame.empty:
        return pd.DataFrame()
    baseline_dice = frame.loc[frame["variant_id"] == "A1", "dice_mean"].iloc[0] if (frame["variant_id"] == "A1").any() else np.nan
    frame["Variant"] = frame["variant_id"].map(lambda variant_id: VARIANT_SPECS[variant_id]["display_name"])
    frame["DeltaDice"] = frame["dice_mean"] - baseline_dice
    frame["Dice"] = [format_mean_std(m, s) for m, s in zip(frame["dice_mean"], frame["dice_std"])]
    frame["IoU"] = [format_mean_std(m, s) for m, s in zip(frame["iou_mean"], frame["iou_std"])]
    frame["HD95"] = [format_mean_std(m, s, precision=1) for m, s in zip(frame["hd95_mean"], frame["hd95_std"])]
    frame["DeltaDice"] = frame["DeltaDice"].map(lambda value: "---" if pd.isna(value) or abs(value) < 1e-12 else f"{value:+.3f}")
    order = {variant_id: idx for idx, variant_id in enumerate(["A1", "A3", "A4", "A5", "A6", "A7", "A2"])}
    frame["__order__"] = frame["variant_id"].map(order)
    return frame.sort_values("__order__")[["variant_id", "Variant", "Dice", "IoU", "HD95", "DeltaDice"]].reset_index(drop=True)


def paired_significance_tests(
    frame: pd.DataFrame,
    reference_variant: str = "A2",
    metrics: list[str] | None = None,
) -> pd.DataFrame:
    metrics = metrics or ["dice", "iou", "hd95"]
    if frame.empty:
        return pd.DataFrame()

    eval_frame = frame[frame["record_type"] == "evaluation"].copy()
    results: list[dict[str, Any]] = []
    for (train_dataset, eval_dataset, split), group in eval_frame.groupby(["train_dataset", "eval_dataset", "split"], dropna=False):
        variants = sorted(group["variant_id"].dropna().unique().tolist())
        if reference_variant not in variants:
            continue
        reference_group = group[group["variant_id"] == reference_variant]
        for variant_id in variants:
            if variant_id == reference_variant:
                continue
            target_group = group[group["variant_id"] == variant_id]
            shared_seeds = sorted(set(reference_group["seed"]) & set(target_group["seed"]))
            for metric in metrics:
                ref_series = reference_group.set_index("seed").loc[shared_seeds][metric].astype(float)
                tgt_series = target_group.set_index("seed").loc[shared_seeds][metric].astype(float)
                if len(shared_seeds) >= 2:
                    statistic, p_value = stats.ttest_rel(ref_series.to_numpy(), tgt_series.to_numpy())
                    test_name = "paired_t_test"
                else:
                    statistic, p_value = (np.nan, np.nan)
                    test_name = "insufficient_samples"
                results.append(
                    {
                        "train_dataset": train_dataset,
                        "eval_dataset": eval_dataset,
                        "split": split,
                        "reference_variant": reference_variant,
                        "other_variant": variant_id,
                        "metric": metric,
                        "reference_mean": float(ref_series.mean()) if len(ref_series) else np.nan,
                        "other_mean": float(tgt_series.mean()) if len(tgt_series) else np.nan,
                        "statistic": float(statistic) if not pd.isna(statistic) else np.nan,
                        "p_value": float(p_value) if not pd.isna(p_value) else np.nan,
                        "n_compared": len(shared_seeds),
                        "significant_0_05": bool(p_value < 0.05) if not pd.isna(p_value) else False,
                        "test_name": test_name,
                    }
                )
    return pd.DataFrame(results)


def dataframe_to_latex(frame: pd.DataFrame) -> str:
    if frame.empty:
        return "% No rows available."
    return frame.to_latex(index=False, escape=False)


local_records = collect_local_records()
local_summary = aggregate_local_results(local_records)

print(f"Collected {len(local_records)} local result rows from {OUTPUT_ROOT}")
if not local_summary.empty:
    display(local_summary.head())

main_table = build_combined_main_table(local_summary)
cross_table = build_cross_dataset_table(local_summary)
ablation_table = build_ablation_table(local_summary, dataset_name="kvasir")
significance_table = paired_significance_tests(local_records, reference_variant="A2")

print("Main comparison table")
display(main_table)

print("Cross-dataset table")
display(cross_table)

print("Ablation table")
display(ablation_table)

print("Significance table")
display(significance_table.head(20))

paper_tables_dir = OUTPUT_ROOT / "paper_tables"
paper_tables_dir.mkdir(parents=True, exist_ok=True)
main_table.to_csv(paper_tables_dir / "main_table.csv", index=False)
cross_table.to_csv(paper_tables_dir / "cross_dataset_table.csv", index=False)
ablation_table.to_csv(paper_tables_dir / "ablation_table.csv", index=False)
significance_table.to_csv(paper_tables_dir / "significance_tests.csv", index=False)
(paper_tables_dir / "main_table.tex").write_text(dataframe_to_latex(main_table), encoding="utf-8")
(paper_tables_dir / "cross_dataset_table.tex").write_text(dataframe_to_latex(cross_table), encoding="utf-8")
(paper_tables_dir / "ablation_table.tex").write_text(dataframe_to_latex(ablation_table), encoding="utf-8")
(paper_tables_dir / "significance_tests.tex").write_text(dataframe_to_latex(significance_table), encoding="utf-8")


## Complexity, Training Dynamics, and Paper Analysis

This section estimates model complexity, plots training curves, produces lesion-size breakdowns, and visualizes the WBS sub-bands for qualitative analysis.


In [ ]:
def estimate_model_complexity(config: dict[str, Any]) -> dict[str, Any]:
    import torch.profiler

    model = WBSNet(config)
    total_params, trainable_params = count_parameters(model)
    model = model.to("cpu").eval()
    dummy = torch.randn(1, 3, config["image_size"][0], config["image_size"][1], device="cpu")
    with torch.no_grad():
        with torch.profiler.profile(
            activities=[torch.profiler.ProfilerActivity.CPU],
            record_shapes=True,
            with_flops=True,
            acc_events=True,
        ) as prof:
            model(dummy)
    flops = float(sum(max(0, int(event.flops)) for event in prof.key_averages()))
    return {
        "variant_id": config["variant_id"],
        "method": local_method_name(config["variant_id"]),
        "params_total": total_params,
        "params_trainable": trainable_params,
        "params_m": total_params / 1e6,
        "gflops": flops / 1e9,
    }


def measure_inference_latency_ms(model: nn.Module, device: torch.device, image_size: tuple[int, int], warmup: int = 10, repeats: int = 30) -> float:
    if device.type != "cuda":
        return float("nan")
    model = model.to(device).eval()
    dummy = torch.randn(1, 3, image_size[0], image_size[1], device=device)
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(dummy)
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(repeats):
            _ = model(dummy)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
    return (elapsed / repeats) * 1000.0


def build_complexity_frame(variant_ids: list[str], train_dataset: str = "kvasir", seed: int = 3407) -> pd.DataFrame:
    records = []
    for variant_id in variant_ids:
        config = build_experiment_config(train_dataset=train_dataset, variant_id=variant_id, seed=seed, run_group="complexity")
        record = estimate_model_complexity(config)
        if variant_id in {"A1", "A2"}:
            model = WBSNet(config)
            record["latency_ms"] = measure_inference_latency_ms(model, DEVICE, tuple(config["image_size"]))
        records.append(record)
    return pd.DataFrame(records).sort_values("params_m").reset_index(drop=True)


def build_complexity_comparison_table(local_complexity: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for method_name, metrics in PAPER_REPORTED_COMPLEXITY.items():
        rows.append({"method": method_name, **metrics})
    frame = pd.DataFrame(rows)
    if not local_complexity.empty:
        for _, row in local_complexity.iterrows():
            mask = frame["method"] == row["method"]
            if mask.any():
                frame.loc[mask, ["params_m", "gflops"]] = [row["params_m"], row["gflops"]]
            else:
                frame = pd.concat(
                    [
                        frame,
                        pd.DataFrame(
                            [
                                {
                                    "method": row["method"],
                                    "params_m": row["params_m"],
                                    "gflops": row["gflops"],
                                    "dice": np.nan,
                                    "hd95": np.nan,
                                }
                            ]
                        ),
                    ],
                    ignore_index=True,
                )
    return frame.sort_values("params_m").reset_index(drop=True)


def plot_complexity_scatter(frame: pd.DataFrame) -> None:
    if frame.empty:
        print("No complexity frame to plot.")
        return
    fig, ax = plt.subplots(figsize=(8, 5))
    for _, row in frame.iterrows():
        ax.scatter(row["params_m"], row["dice"], s=80)
        ax.text(row["params_m"] + 0.2, row["dice"] + 0.0005, row["method"], fontsize=9)
    ax.set_xlabel("Parameters (M)")
    ax.set_ylabel("Dice")
    ax.set_title("Dice vs Parameters")
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()


def parse_result_context(path: str | Path, root: str | Path = OUTPUT_ROOT) -> dict[str, str]:
    path = Path(path)
    root = Path(root)
    context = {
        "run_group": "unknown",
        "train_dataset": "unknown",
        "variant_id": "unknown",
        "seed_dir": "unknown",
        "run_name": "unknown",
    }
    try:
        relative_parts = path.relative_to(root).parts
    except ValueError:
        relative_parts = path.parts
    if len(relative_parts) >= 5:
        context.update(
            {
                "run_group": relative_parts[0],
                "train_dataset": relative_parts[1],
                "variant_id": relative_parts[2],
                "seed_dir": relative_parts[3],
                "run_name": relative_parts[4],
            }
        )
    return context


def load_history_frames(root: str | Path = OUTPUT_ROOT) -> pd.DataFrame:
    rows: list[pd.DataFrame] = []
    fallback_columns = [
        "epoch",
        "train_total_loss",
        "val_total_loss",
        "train_dice",
        "val_dice",
        "encoder_lr",
        "decoder_lr",
        "run_group",
        "train_dataset",
        "variant_id",
        "seed_dir",
        "run_name",
        "source",
    ]
    for path in Path(root).rglob("metrics.csv"):
        frame = pd.read_csv(path)
        context = parse_result_context(path, root)
        for key, value in context.items():
            frame[key] = value
        frame["source"] = str(path)
        rows.append(frame)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=fallback_columns)


def plot_training_dynamics(history_frame: pd.DataFrame, train_dataset: str = "kvasir", variant_ids: list[str] | None = None) -> None:
    variant_ids = variant_ids or ["A1", "A2"]
    required_columns = {"epoch", "variant_id", "train_dataset", "val_dice", "val_total_loss"}
    if history_frame.empty or not required_columns.issubset(history_frame.columns):
        print(f"No history available for {train_dataset} and variants {variant_ids}.")
        return
    frame = history_frame[history_frame["train_dataset"] == train_dataset].copy()
    frame = frame[frame["variant_id"].isin(variant_ids)]
    if frame.empty:
        print(f"No history available for {train_dataset} and variants {variant_ids}.")
        return

    summary = (
        frame.groupby(["epoch", "variant_id"])[["val_dice", "val_total_loss"]]
        .agg(["mean", "std"])
        .reset_index()
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for variant_id in variant_ids:
        subset = summary[summary["epoch"].notna() & (summary["variant_id"] == variant_id)]
        if subset.empty:
            continue
        dice_mean = subset[("val_dice", "mean")].to_numpy()
        dice_std = subset[("val_dice", "std")].fillna(0.0).to_numpy()
        loss_mean = subset[("val_total_loss", "mean")].to_numpy()
        loss_std = subset[("val_total_loss", "std")].fillna(0.0).to_numpy()
        epochs = subset["epoch"].to_numpy()

        axes[0].plot(epochs, dice_mean, label=variant_id)
        axes[0].fill_between(epochs, dice_mean - dice_std, dice_mean + dice_std, alpha=0.15)
        axes[1].plot(epochs, loss_mean, label=variant_id)
        axes[1].fill_between(epochs, loss_mean - loss_std, loss_mean + loss_std, alpha=0.15)

    axes[0].set_title(f"{train_dataset} validation Dice")
    axes[1].set_title(f"{train_dataset} validation total loss")
    for axis in axes:
        axis.set_xlabel("Epoch")
        axis.grid(True, alpha=0.25)
        axis.legend()
    plt.tight_layout()
    plt.show()


def collect_sample_metric_frames(root: str | Path = OUTPUT_ROOT) -> pd.DataFrame:
    frames = []
    fallback_columns = [
        "sample_id",
        "dice",
        "iou",
        "precision",
        "recall",
        "accuracy",
        "specificity",
        "hd95",
        "target_area_ratio",
        "pred_area_ratio",
        "image_path",
        "mask_path",
        "boundary_path",
        "run_group",
        "train_dataset",
        "variant_id",
        "seed_dir",
        "run_name",
        "eval_dataset",
        "split",
        "source",
    ]
    for path in Path(root).rglob("sample_metrics.csv"):
        frame = pd.read_csv(path)
        context = parse_result_context(path, root)
        eval_name = path.parent.name
        if "_" in eval_name:
            eval_dataset, split = eval_name.rsplit("_", 1)
        else:
            eval_dataset, split = eval_name, "unknown"
        for key, value in context.items():
            frame[key] = value
        frame["eval_dataset"] = eval_dataset
        frame["split"] = split
        frame["source"] = str(path)
        frames.append(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=fallback_columns)


def lesion_size_breakdown(sample_frame: pd.DataFrame, train_dataset: str = "kvasir", eval_dataset: str | None = None, variants: list[str] | None = None) -> pd.DataFrame:
    variants = variants or list(RESEARCH_CONFIG["size_breakdown_reference_variants"])
    eval_dataset = eval_dataset or train_dataset
    required_columns = {"train_dataset", "eval_dataset", "variant_id", "sample_id", "target_area_ratio", "dice"}
    if sample_frame.empty or not required_columns.issubset(sample_frame.columns):
        return pd.DataFrame()

    frame = sample_frame[
        (sample_frame["train_dataset"] == train_dataset)
        & (sample_frame["eval_dataset"] == eval_dataset)
        & (sample_frame["variant_id"].isin(variants))
    ].copy()
    if frame.empty:
        return pd.DataFrame()

    area_reference = frame[frame["variant_id"] == variants[0]][["sample_id", "target_area_ratio"]].drop_duplicates("sample_id")
    if len(area_reference) < 2:
        return pd.DataFrame()

    quartile_labels = ["Q1", "Q2", "Q3", "Q4"]
    quantile_count = min(4, len(area_reference))
    ranked_area = area_reference["target_area_ratio"].rank(method="first")
    area_reference["quartile"] = pd.qcut(ranked_area, q=quantile_count, labels=quartile_labels[:quantile_count])
    frame = frame.merge(area_reference[["sample_id", "quartile"]], on="sample_id", how="left")
    summary = (
        frame.groupby(["variant_id", "quartile"])["dice"]
        .mean()
        .reset_index()
        .pivot(index="variant_id", columns="quartile", values="dice")
        .reset_index()
    )
    for quartile in quartile_labels:
        if quartile not in summary.columns:
            summary[quartile] = np.nan
    summary["Method"] = summary["variant_id"].map(lambda variant_id: VARIANT_SPECS[variant_id]["paper_name"])
    ordered_columns = ["Method", "Q1", "Q2", "Q3", "Q4"]
    return summary[ordered_columns]


@torch.no_grad()
def visualize_first_wbs_subbands(run_summary_path: str | Path | None = None, sample_index: int = 0) -> None:
    if run_summary_path is None:
        candidates = sorted(OUTPUT_ROOT.rglob("run_summary.json"))
        if not candidates:
            print("No run_summary.json files found.")
            return
        run_summary_path = candidates[0]

    summary = load_json(run_summary_path)
    config_path = Path(run_summary_path).parent / "resolved_config.json"
    if not config_path.exists():
        raise FileNotFoundError(f"Missing resolved_config.json next to {run_summary_path}")
    config = load_json(config_path)
    model = WBSNet(config).to(DEVICE)
    load_checkpoint(summary["best_checkpoint"], model, map_location=DEVICE)
    model.eval()

    dataset_name = config["train_dataset"]
    split_name = resolve_eval_split(dataset_name)
    dataset = ProcessedWBSNetDataset(PROCESSED_ROOT, dataset_name, split_name, train=False, config=config)
    sample = dataset[sample_index]
    image = sample["image"].unsqueeze(0).to(DEVICE)
    mask = sample["mask"]

    stem, _, _, _, _ = model.encoder(image)
    debug = model.skip_modules[0].debug_subbands(stem)
    if "ll" not in debug:
        print("The selected model does not use the wavelet decomposition at the first skip.")
        return

    ll = debug["ll"][0].mean(dim=0).detach().cpu().numpy()
    lh = debug["lh"][0].mean(dim=0).detach().cpu().numpy()
    hl = debug["hl"][0].mean(dim=0).detach().cpu().numpy()
    hh = debug["hh"][0].mean(dim=0).detach().cpu().numpy()
    boundary_map = torch.sigmoid(debug["boundary_logits"][0, 0]).detach().cpu().numpy()
    image_np = denormalize_image_tensor(sample["image"], config["normalize_mean"], config["normalize_std"])
    mask_np = tensor_to_mask_uint8(mask)

    fig, axes = plt.subplots(1, 7, figsize=(22, 4))
    axes[0].imshow(image_np)
    axes[0].set_title("Input")
    axes[1].imshow(mask_np, cmap="gray")
    axes[1].set_title("Mask")
    axes[2].imshow(ll, cmap="viridis")
    axes[2].set_title("LL")
    axes[3].imshow(lh, cmap="viridis")
    axes[3].set_title("LH")
    axes[4].imshow(hl, cmap="viridis")
    axes[4].set_title("HL")
    axes[5].imshow(hh, cmap="viridis")
    axes[5].set_title("HH")
    axes[6].imshow(boundary_map, cmap="magma")
    axes[6].set_title("HFBA boundary")
    for axis in axes:
        axis.axis("off")
    plt.tight_layout()
    plt.show()


local_complexity = build_complexity_frame(["A1", "A2", "A3", "A4", "A5", "A6", "A7"], train_dataset="kvasir", seed=3407)
complexity_table = build_complexity_comparison_table(local_complexity)
display(local_complexity)
display(complexity_table)
plot_complexity_scatter(complexity_table)

history_frame = load_history_frames()
if not history_frame.empty:
    plot_training_dynamics(history_frame, train_dataset="kvasir", variant_ids=["A1", "A2"])
else:
    print("No training-history CSV files found yet.")

sample_metric_frame = collect_sample_metric_frames()
size_breakdown = lesion_size_breakdown(sample_metric_frame, train_dataset="kvasir", eval_dataset="kvasir", variants=["A1", "A2"])
if not size_breakdown.empty:
    display(size_breakdown)
else:
    print("No per-sample evaluation metrics found for lesion-size breakdown yet.")

if list(OUTPUT_ROOT.rglob("run_summary.json")):
    visualize_first_wbs_subbands()
else:
    print("No completed run found yet for sub-band visualization.")


## What This Notebook Produces for the Paper

After the full workflow finishes, you should have:
- checkpoints and per-epoch histories for each run
- evaluation JSON files and per-sample metrics
- qualitative prediction panels and contact sheets
- local aggregation tables for A1 to A7
- significance-test CSV and LaTeX tables
- complexity estimates and Dice-vs-parameter plots
- lesion-size breakdown tables
- wavelet sub-band visualizations and training-dynamics plots

These artifacts are enough to support the main experimental sections of the paper while keeping all project logic inside the notebook.
